In [ ]:
# ============================================================
# SnowBallScan — Kaggle Experiment Code v10 (FP16, n=200, robust dataset loading)
# Sequential Hallucination Propagation in Quantized SLMs
# ============================================================
# FIXES vs v2:
#   - Context format fixed: pass only last response summary
#     (not full raw accumulation — prevents sentence completion loops)
#   - FreshQA: additional dataset IDs + manual CSV fallback
#   - Checkpoint: auto-detects stale v1 rows (looping responses)
#     and skips/cleans them before running
#   - Phi-2: explicit bitsandbytes version check before loading
#   - Response quality check: detects repetition loops, retries once
#
# USAGE:
#   1. DELETE old checkpoint.json before first run of v3
#   2. SAMPLE_SIZE = 1  → quick test
#   3. SAMPLE_SIZE = 5  → sanity check
#   4. SAMPLE_SIZE = 200 → full run
# ============================================================

import os, json, random, warnings, re, sys
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

# ── BITSANDBYTES NOTE ─────────────────────────────────────────────────────────
# Kaggle CUDA 12.8 has no precompiled bitsandbytes binary.
# This version runs FP16 only — INT8/INT4 noted as future work in paper.
print("FP16-only mode (bitsandbytes not available on CUDA 12.8 Kaggle environment).")
BNB_AVAILABLE = False

# ── CONFIG ────────────────────────────────────────────────────────────────────
SAMPLE_SIZE   = 200       # Full run
CHAIN_LENGTH  = 5
PAI_THRESHOLD = 1.5
SEED          = 42
CHECKPOINT    = "checkpoint_v10.json"  # v10 full run — clean start
RESULTS_XLSX  = "snowballscan_results_v10.xlsx"

random.seed(SEED)
np.random.seed(SEED)

MODELS = {
    "phi2":      "microsoft/phi-2",
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "qwen25":    "Qwen/Qwen2.5-1.5B-Instruct",
}

PRECISIONS = ["fp16"]  # INT8/INT4 not available on Kaggle CUDA 12.8

DATASETS = ["truthfulqa", "freshqa", "halueval"]

# ── INSTALL DEPS ──────────────────────────────────────────────────────────────
# no installs needed — FP16 only
# bitsandbytes not used — FP16 only
os.system("pip install -q auto-gptq optimum autoawq rank_bm25 sentence-transformers datasets openpyxl")

# bitsandbytes not used in FP16-only mode

print("Dependencies ready.\n")

# ── IMPORTS ───────────────────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

# ── REPETITION DETECTOR ───────────────────────────────────────────────────────
def is_repetition_loop(text, threshold=0.5):
    """
    Returns True if response is a repetition loop.
    Checks: repeated n-grams, numbered list pattern, same phrase >3 times.
    """
    if not text or len(text.strip()) < 5:
        return True
    # Numbered list pattern (1. 2. 3. repeating same content)
    if re.search(r'\d+\.\s+\w+.*\d+\.\s+\w+.*\d+\.\s+\w+', text, re.DOTALL):
        # Check if the content between numbers is similar
        items = re.findall(r'\d+\.\s+(.{10,50})', text)
        if len(items) >= 3:
            # Check similarity — if first 20 chars repeat
            prefixes = [i[:20].lower() for i in items]
            if len(set(prefixes)) < len(prefixes) * threshold:
                return True
    # Same phrase repeated 3+ times
    words = text.lower().split()
    if len(words) > 10:
        trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
        from collections import Counter
        counts = Counter(trigrams)
        if counts.most_common(1)[0][1] >= 4:
            return True
    return False

# ── PROMPT TEMPLATES ─────────────────────────────────────────────────────────
def format_prompt(model_name, question, prior_response=None):
    """
    model_name: key from MODELS dict
    question: original question
    prior_response: LAST response only (truncated to 100 chars) — not full history
    """
    # Context hint: only last response, briefly
    if prior_response:
        ctx = prior_response[:120].strip()
        context_hint = f"Your previous answer was: \"{ctx}\". Now answer again more carefully."
    else:
        context_hint = None

    if model_name == "tinyllama":
        sys_msg = "You are a factual assistant. Answer questions with a single sentence. Do not repeat yourself."
        if context_hint:
            user_msg = f"{context_hint}\n\nQuestion: {question}\nProvide a single factual sentence answer:"
        else:
            user_msg = f"Question: {question}\nProvide a single factual sentence answer:"
        return f"<|system|>\n{sys_msg}</s>\n<|user|>\n{user_msg}</s>\n<|assistant|>\n"

    elif model_name == "phi2":
        if context_hint:
            return (f"Instruct: {context_hint} Answer the following question in one sentence.\n"
                    f"Question: {question}\nOutput:")
        return f"Instruct: Answer the following question in one sentence.\nQuestion: {question}\nOutput:"

    elif model_name == "qwen25":
        sys_msg = "You are a factual assistant. Answer each question with one clear, concise sentence."
        if context_hint:
            user_msg = f"{context_hint}\n\nQuestion: {question}"
        else:
            user_msg = question
        return (f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n")

    else:
        if context_hint:
            return f"{context_hint}\nQuestion: {question}\nAnswer in one sentence:"
        return f"Question: {question}\nAnswer in one sentence:"

# ── CHECKPOINT HELPERS ────────────────────────────────────────────────────────
def load_checkpoint():
    if Path(CHECKPOINT).exists():
        with open(CHECKPOINT) as f:
            data = json.load(f)
        print(f"Resuming checkpoint: {len(data['rows'])} rows, {len(data['done_keys'])} chains done.")
        return data
    print("No checkpoint found — starting fresh.")
    return {"rows": [], "done_keys": []}

def save_checkpoint(data):
    with open(CHECKPOINT, "w") as f:
        json.dump(data, f, indent=2)

def make_key(model_name, precision, dataset, q_idx):
    return f"{model_name}|{precision}|{dataset}|{q_idx}"

# ── DATASET LOADERS ───────────────────────────────────────────────────────────
def load_datasets_all(sample_size):
    """
    Load all three datasets from uploaded Kaggle input files.
    Paths: /kaggle/input/snowballscan-dataset/
      - TruthfulQA.csv
      - freshqa.csv
      - halueval.json
    Falls back to HuggingFace if files not found.
    """
    import pandas as _pd, json as _json, random as _r
    out = {}
    BASE = "/kaggle/input/datasets/samuelstephen77/snowballscan-dataset"


    # ── TruthfulQA ────────────────────────────────────────────────────────────
    print("Loading TruthfulQA...")
    tqa_path = None
    for candidate in [
        f"{BASE}/TruthfulQA.csv",
        f"{BASE}/truthfulqa.csv",
        f"{BASE}/TruthfulQA.CSV",
        "/kaggle/input/snowballscan-dataset/TruthfulQA.csv",
    ]:
        if Path(candidate).exists():
            tqa_path = candidate
            print(f"  Found TruthfulQA at: {candidate}")
            break
    if tqa_path is None:
        import glob
        matches = glob.glob('/kaggle/input/**/TruthfulQA.csv', recursive=True) +                   glob.glob('/kaggle/input/**/truthfulqa.csv', recursive=True)
        if matches:
            tqa_path = matches[0]
            print(f"  Found TruthfulQA via glob: {tqa_path}")
    tqa_loaded = False
    if tqa_path is not None:
        try:
            df = _pd.read_csv(tqa_path)
            # Standard TruthfulQA columns: Type, Category, Question, Best Answer,
            # Correct Answers, Incorrect Answers, Source
            # Normalise column names
            df.columns = [c.strip() for c in df.columns]
            q_col   = next(c for c in df.columns if 'question' in c.lower())
            a_col   = next(c for c in df.columns if 'best' in c.lower() and 'answer' in c.lower())
            cor_col = next((c for c in df.columns if 'correct' in c.lower() and 'answer' in c.lower()), None)
            df = df[df[q_col].notna()].reset_index(drop=True)
            df = df.sample(min(sample_size, len(df)), random_state=SEED)
            items = []
            for _, row in df.iterrows():
                q    = str(row[q_col]).strip()
                gold = str(row[a_col]).strip()
                if cor_col:
                    # Correct Answers column may be semicolon-separated
                    corpus = [a.strip() for a in str(row[cor_col]).split(';') if a.strip() and a.strip() != 'nan']
                    corpus = corpus if corpus else [gold]
                else:
                    corpus = [gold]
                if q and gold:
                    items.append({"question": q, "gold": gold, "corpus": corpus})
            if items:
                out["truthfulqa"] = items
                print(f"  TruthfulQA loaded from CSV: {len(items)} samples ✓")
                tqa_loaded = True
        except Exception as e:
            print(f"  TruthfulQA CSV failed: {e}")

    if not tqa_loaded:
        print("  Falling back to HuggingFace TruthfulQA...")
        tqa = load_dataset("truthful_qa", "generation", split="validation")
        tqa = tqa.shuffle(seed=SEED).select(range(min(sample_size, len(tqa))))
        out["truthfulqa"] = [
            {"question": r["question"], "gold": r["best_answer"],
             "corpus": r["correct_answers"] if r["correct_answers"] else [r["best_answer"]]}
            for r in tqa
        ]
        print(f"  TruthfulQA HF: {len(out['truthfulqa'])} samples ✓")

    # ── FreshQA ───────────────────────────────────────────────────────────────
    print("Loading FreshQA...")
    # Try multiple path variants — Kaggle is case-sensitive
    fqa_path = None
    for candidate in [
        f"{BASE}/freshqa.csv",
        f"{BASE}/freshqa.CSV",
        f"{BASE}/FreshQA.csv",
        "/kaggle/input/snowballscan-dataset/freshqa.csv",
        "/kaggle/input/snowballscandataset/freshqa.csv",
    ]:
        if Path(candidate).exists():
            fqa_path = candidate
            print(f"  Found FreshQA at: {candidate}")
            break
    if fqa_path is None:
        # Last resort: search all kaggle input
        import glob
        matches = glob.glob('/kaggle/input/**/freshqa.csv', recursive=True) +                   glob.glob('/kaggle/input/**/freshqa.CSV', recursive=True)
        if matches:
            fqa_path = matches[0]
            print(f"  Found FreshQA via glob: {fqa_path}")
    freshqa_loaded = False
    if fqa_path is not None:
        try:
            df = _pd.read_csv(fqa_path, skiprows=2, header=None)
            df.columns = ['id','split','question','effective_year','next_review',
                          'false_premise','num_hops','fact_type','source',
                          'answer_0','answer_1','answer_2','answer_3','answer_4',
                          'answer_5','answer_6','answer_7','answer_8','answer_9','note']
            df = df[df['question'].notna()]
            df = df[df['false_premise'].astype(str).str.upper() != 'TRUE']
            df = df[df['question'] != 'question'].reset_index(drop=True)
            df = df.sample(min(sample_size, len(df)), random_state=SEED)
            items = []
            for _, row in df.iterrows():
                q = str(row['question']).strip()
                ans_cols = [f'answer_{i}' for i in range(10)]
                answers = [str(row[c]).strip() for c in ans_cols
                           if c in row and str(row[c]).strip() not in ['', 'nan', 'NaN']]
                if q and answers:
                    items.append({"question": q, "gold": answers[0], "corpus": answers})
            if items:
                out["freshqa"] = items
                print(f"  FreshQA loaded from CSV: {len(items)} samples ✓")
                freshqa_loaded = True
        except Exception as e:
            print(f"  FreshQA CSV failed: {e}")

    if not freshqa_loaded:
        print("  FreshQA CSV not found — using TruthfulQA subset as proxy.")
        out["freshqa"] = out["truthfulqa"][:sample_size]

    # ── HaluEval ──────────────────────────────────────────────────────────────
    print("Loading HaluEval...")
    hqa_path = None
    for candidate in [
        f"{BASE}/halueval.json",
        f"{BASE}/HaluEval.json",
        "/kaggle/input/snowballscan-dataset/halueval.json",
    ]:
        if Path(candidate).exists():
            hqa_path = candidate
            print(f"  Found HaluEval at: {candidate}")
            break
    if hqa_path is None:
        import glob
        matches = glob.glob('/kaggle/input/**/halueval.json', recursive=True) +                   glob.glob('/kaggle/input/**/HaluEval.json', recursive=True)
        if matches:
            hqa_path = matches[0]
            print(f"  Found HaluEval via glob: {hqa_path}")
    halueval_loaded = False
    if hqa_path is not None:
        try:
            with open(hqa_path) as f:
                raw = _json.load(f)
            # HaluEval QA format: list of {question, right_answer, hallucinated_answer}
            if isinstance(raw, dict):
                # May be wrapped: {"data": [...]}
                raw = raw.get("data", raw.get("qa", list(raw.values())[0]))
            _r.seed(SEED); _r.shuffle(raw)
            items = []
            for r in raw[:sample_size]:
                q    = str(r.get("question","")).strip()
                gold = str(r.get("right_answer","")).strip()
                hall = str(r.get("hallucinated_answer","")).strip()
                if q and gold:
                    items.append({
                        "question":     q,
                        "gold":         gold,
                        "hallucinated": hall if hall else None,
                        "corpus":       [gold]
                    })
            if items:
                out["halueval"] = items
                print(f"  HaluEval loaded from JSON: {len(items)} samples ✓")
                halueval_loaded = True
        except Exception as e:
            print(f"  HaluEval JSON failed: {e}")

    if not halueval_loaded:
        print("  Falling back to HuggingFace HaluEval...")
        try:
            hqa = load_dataset("pminervini/HaluEval", "qa", split="data")
            hqa = hqa.shuffle(seed=SEED).select(range(min(sample_size, len(hqa))))
            out["halueval"] = [
                {"question": r["question"], "gold": r["right_answer"],
                 "hallucinated": r["hallucinated_answer"], "corpus": [r["right_answer"]]}
                for r in hqa
            ]
            print(f"  HaluEval HF: {len(out['halueval'])} samples ✓")
        except Exception as e:
            print(f"  HaluEval HF failed ({e}) — using TruthfulQA subset.")
            out["halueval"] = out["truthfulqa"][:sample_size]

    print(f"\nDatasets ready: { {k: len(v) for k,v in out.items()} }\n")
    return out

# ── MODEL LOADER ──────────────────────────────────────────────────────────────
def load_model(model_name, model_id, precision):
    print(f"Loading {model_name} @ {precision}...")
    torch.manual_seed(SEED)

    kwargs = dict(trust_remote_code=True, device_map="auto")

    # Pre-load config and patch pad_token_id — fixes Phi-2 PhiConfig error
    # which occurs inside from_pretrained before we can patch anything
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
        config.pad_token_id = 2  # eos_token_id for Phi-2; safe default for all models

    if precision == "fp16":
        model = AutoModelForCausalLM.from_pretrained(
            model_id, config=config, dtype=torch.float16, **kwargs)

    elif precision == "int8":
        if not BNB_AVAILABLE:
            raise RuntimeError("bitsandbytes INT8 not available — skipping")
        bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, config=config, quantization_config=bnb_cfg, **kwargs)

    elif precision == "int4":
        if not BNB_AVAILABLE:
            raise RuntimeError("bitsandbytes INT4 not available — skipping")
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id, config=config, quantization_config=bnb_cfg, **kwargs)

    else:
        raise ValueError(f"Unknown precision: {precision}")

    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token     = tok.eos_token
        tok.pad_token_id  = tok.eos_token_id
    if not hasattr(model.config, 'pad_token_id') or model.config.pad_token_id is None:
        model.config.pad_token_id = tok.pad_token_id

    model.eval()
    mem = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
    print(f"  {model_name} @ {precision} ready. GPU: {mem:.2f}GB\n")
    return model, tok

# ── GENERATION ────────────────────────────────────────────────────────────────
def generate_response(model, tok, model_name, question,
                      prior_response=None, max_new_tokens=100):
    prompt = format_prompt(model_name, question, prior_response)
    inputs = tok(prompt, return_tensors="pt", truncation=True,
                 max_length=768, padding=False).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.3,   # mild penalty to reduce loops
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    new = out[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(new, skip_special_tokens=True).strip()

    # Take first sentence only
    for sep in [". ", ".\n", "\n"]:
        if sep in text:
            text = text.split(sep)[0].strip() + "."
            break

    # Detect loop — retry once with higher repetition penalty
    if is_repetition_loop(text):
        inputs2 = tok(format_prompt(model_name, question, None),
                      return_tensors="pt", truncation=True,
                      max_length=512, padding=False).to(model.device)
        with torch.no_grad():
            out2 = model.generate(
                **inputs2,
                max_new_tokens=80,
                do_sample=False,
                temperature=1.0,
                repetition_penalty=1.8,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id,
            )
        new2 = out2[0][inputs2["input_ids"].shape[1]:]
        text2 = tok.decode(new2, skip_special_tokens=True).strip()
        for sep in [". ", ".\n", "\n"]:
            if sep in text2:
                text2 = text2.split(sep)[0].strip() + "."
                break
        if not is_repetition_loop(text2):
            text = text2

    return text if text else "No answer generated."

# ── HALLUCINATION DETECTION ───────────────────────────────────────────────────
def load_entailment_model():
    print("Loading entailment model (DeBERTa-v3-base, ~1.2GB VRAM)...")
    m = CrossEncoder("cross-encoder/nli-deberta-v3-base")
    print("  Entailment model ready.\n")
    return m

def is_hallucinated(entail_model, response, gold):
    if not response or not gold:
        return 1
    try:
        scores = entail_model.predict([(gold, response)])
        e_score = float(scores[0][2]) if scores[0].shape[0]==3 else float(scores[0][-1])
    except Exception:
        e_score = 0.0
    gold_tok = set(gold.lower().split())
    resp_tok = set(response.lower().split())
    overlap  = len(gold_tok & resp_tok) / max(len(gold_tok), 1)
    return 0 if (e_score > 0.4 or overlap > 0.3) else 1

# ── BM25 RETRIEVAL ────────────────────────────────────────────────────────────
def build_bm25(corpus):
    tok = [d.lower().split() for d in corpus if d]
    return BM25Okapi(tok) if tok else None

def retrieve_correction(bm25, corpus, query):
    if not bm25 or not corpus:
        return corpus[0] if corpus else ""
    scores = bm25.get_scores(query.lower().split())
    return corpus[int(np.argmax(scores))]

# ── CHAIN RUNNER ─────────────────────────────────────────────────────────────
def run_chain(model, tok, model_name, entail_model,
              question, gold, corpus,
              seeded_response=None,
              apply_mitigation=False,
              non_propagating=False):
    """
    KEY CHANGE vs v2:
    Context passed to each step = ONLY the last response (truncated to 120 chars),
    NOT the full accumulated history. This prevents sentence-completion loops
    while still propagating hallucination signal.
    """
    bm25          = build_bm25(corpus)
    last_response = None   # only last response passed as context
    results       = []

    for k in range(1, CHAIN_LENGTH + 1):
        # For non-propagating: never pass prior context
        ctx = None if non_propagating else last_response

        # Generate
        if k == 1 and seeded_response is not None:
            resp = seeded_response
        else:
            resp = generate_response(model, tok, model_name, question,
                                     prior_response=ctx)

        # Detect hallucination
        h = is_hallucinated(entail_model, resp, gold)

        # Mitigation at steps 2 and 4
        if apply_mitigation and k in [2, 4] and h == 1:
            correction = retrieve_correction(bm25, corpus, question)
            if correction:
                resp = correction
                h    = 0

        results.append({
            "step":         k,
            "response":     resp[:300],
            "hallucinated": h,
            "loop_detected": int(is_repetition_loop(resp)),
        })

        # Update last_response for next step (propagating only)
        if not non_propagating:
            last_response = resp

    return results

# ── PAI & ONSET ───────────────────────────────────────────────────────────────
def compute_pai(h_rates):
    """
    PAI(k) = H(k) / H(1).
    If H(1) == 0: model had no hallucinations at step 1.
    In this case PAI is undefined — return None to flag it separately.
    If H(k) > 0 but H(1) == 0: indicates emergence (new hallucinations appearing),
    which we cap at 2.0 as a conservative upper bound.
    """
    if not h_rates:
        return [1.0] * len(h_rates)
    if h_rates[0] == 0:
        # No hallucination at step 1 — check if any emerge later
        return [1.0 if h == 0 else 2.0 for h in h_rates]
    return [round(h / h_rates[0], 4) for h in h_rates]

def compute_onset(pai_vals, threshold=PAI_THRESHOLD):
    for i, v in enumerate(pai_vals):
        if v >= threshold:
            return i + 1
    return None

# ── MAIN EXPERIMENT ───────────────────────────────────────────────────────────
def run_experiment():
    ckpt      = load_checkpoint()
    done_keys = set(ckpt["done_keys"])
    rows      = ckpt["rows"]

    datasets     = load_datasets_all(SAMPLE_SIZE)
    entail_model = load_entailment_model()

    for model_name, model_id in MODELS.items():
        for precision in PRECISIONS:

            try:
                model, tok = load_model(model_name, model_id, precision)
            except Exception as e:
                print(f"SKIP {model_name} {precision}: {e}\n")
                continue

            for dataset_name in DATASETS:
                questions = datasets[dataset_name]

                sf       = {k: [] for k in range(1, CHAIN_LENGTH+1)}  # propagating
                sf_m     = {k: [] for k in range(1, CHAIN_LENGTH+1)}  # mitigated
                sf_np    = {k: [] for k in range(1, CHAIN_LENGTH+1)}  # non-propagating

                for q_idx, qdata in enumerate(questions):
                    key = make_key(model_name, precision, dataset_name, q_idx)
                    if key in done_keys:
                        print(f"  SKIP (done): {key}")
                        continue

                    print(f"  [{model_name}|{precision}|{dataset_name}|q{q_idx}] "
                          f"Q: {qdata['question'][:60]}...")

                    q         = qdata["question"]
                    gold      = qdata["gold"]
                    corpus    = qdata["corpus"]
                    seed_resp = qdata.get("hallucinated", None)

                    try:
                        # 1. Propagating
                        chain = run_chain(model, tok, model_name, entail_model,
                                          q, gold, corpus,
                                          seeded_response=seed_resp,
                                          apply_mitigation=False)

                        # 2. Mitigated
                        chain_m = run_chain(model, tok, model_name, entail_model,
                                            q, gold, corpus,
                                            seeded_response=seed_resp,
                                            apply_mitigation=True)

                        # 3. Non-propagating baseline
                        chain_np = run_chain(model, tok, model_name, entail_model,
                                             q, gold, corpus,
                                             seeded_response=None,
                                             apply_mitigation=False,
                                             non_propagating=True)

                        # Accumulate flags
                        for s in chain:
                            sf[s["step"]].append(s["hallucinated"])
                        for s in chain_m:
                            sf_m[s["step"]].append(s["hallucinated"])
                        for s in chain_np:
                            sf_np[s["step"]].append(s["hallucinated"])

                        # Store rows
                        for ct, ch in [("propagating", chain),
                                       ("mitigated",   chain_m),
                                       ("non_propagating", chain_np)]:
                            for s in ch:
                                rows.append({
                                    "model":         model_name,
                                    "precision":     precision,
                                    "dataset":       dataset_name,
                                    "q_idx":         q_idx,
                                    "chain_type":    ct,
                                    "step":          s["step"],
                                    "hallucinated":  s["hallucinated"],
                                    "loop_detected": s["loop_detected"],
                                    "response":      s["response"],
                                })

                        done_keys.add(key)
                        ckpt["rows"]      = rows
                        ckpt["done_keys"] = list(done_keys)
                        save_checkpoint(ckpt)

                        flags = [s["hallucinated"] for s in chain]
                        loops = sum(s["loop_detected"] for s in chain)
                        print(f"    H-flags: {flags}  Loops: {loops}/5")

                    except Exception as e:
                        print(f"    ERROR: {e}")
                        import traceback; traceback.print_exc()
                        continue

                # PAI summary
                h  = [np.mean(sf[k])    if sf[k]    else 0.0 for k in range(1, CHAIN_LENGTH+1)]
                hm = [np.mean(sf_m[k])  if sf_m[k]  else 0.0 for k in range(1, CHAIN_LENGTH+1)]
                hn = [np.mean(sf_np[k]) if sf_np[k] else 0.0 for k in range(1, CHAIN_LENGTH+1)]

                pai  = compute_pai(h)
                paim = compute_pai(hm)
                pain = compute_pai(hn)
                onset = compute_onset(pai)

                print(f"\n  ── [{model_name}|{precision}|{dataset_name}] ──")
                for k in range(CHAIN_LENGTH):
                    print(f"     Step {k+1}: H={h[k]:.3f}  PAI={pai[k]:.3f}")
                print(f"  PAI@5: {pai[-1]:.3f} | Mitigated: {paim[-1]:.3f} | Non-prop: {pain[-1]:.3f}")
                print(f"  Onset: {'Step '+str(onset) if onset else 'Not Reached (∞)'}\n")

            del model, tok
            torch.cuda.empty_cache()
            print(f"GPU cleared: {model_name} {precision}\n")

    print("All chains done. Building XLSX...")
    build_results_xlsx(rows)

# ── RESULTS XLSX ──────────────────────────────────────────────────────────────
def build_results_xlsx(rows):
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment

    if not rows:
        print("No rows to save.")
        return

    df = pd.DataFrame(rows)
    wb = openpyxl.Workbook()

    # Sheet 1: Raw
    ws1 = wb.active; ws1.title = "Raw Chains"
    _ws(ws1, df)

    # Sheet 2: PAI by step
    prop = df[df["chain_type"]=="propagating"]
    summ = []
    for (m,p,d), g in prop.groupby(["model","precision","dataset"]):
        h1 = g[g["step"]==1]["hallucinated"].mean() if len(g[g["step"]==1]) else 0
        for step in range(1, CHAIN_LENGTH+1):
            sg = g[g["step"]==step]
            hr = sg["hallucinated"].mean() if len(sg) else 0.0
            loops = sg["loop_detected"].mean() if "loop_detected" in sg.columns and len(sg) else 0.0
            pai = round(hr/h1, 4) if h1>0 else 1.0
            summ.append({"model":m,"precision":p,"dataset":d,"step":step,
                         "H_rate":round(hr,4),"PAI":pai,"loop_rate":round(loops,4),"n":len(sg)})
    ws2 = wb.create_sheet("PAI by Step"); _ws(ws2, pd.DataFrame(summ))

    # Sheet 3: PAI@5
    ds = pd.DataFrame(summ)
    ws3 = wb.create_sheet("PAI at Step 5")
    _ws(ws3, ds[ds["step"]==5][["model","precision","dataset","H_rate","PAI","loop_rate","n"]].reset_index(drop=True))

    # Sheet 4: Onset
    onset_rows = []
    for (m,p,d), g in ds.groupby(["model","precision","dataset"]):
        g = g.sort_values("step")
        pais = g["PAI"].tolist()
        onset = compute_onset(pais)
        onset_rows.append({
            "model":m,"precision":p,"dataset":d,
            "onset_step": f"Step {onset}" if onset else "Not Reached (∞)",
            "PAI_step1":  round(pais[0],4) if pais else None,
            "PAI_step5":  round(pais[-1],4) if pais else None,
        })
    ws4 = wb.create_sheet("Onset Steps"); _ws(ws4, pd.DataFrame(onset_rows))

    # Sheet 5: Mitigation
    mit_rows = []
    for (m,p,d), g in df.groupby(["model","precision","dataset"]):
        for ct in ["propagating","mitigated","non_propagating"]:
            sg = g[(g["chain_type"]==ct)&(g["step"]==5)]
            hr = round(sg["hallucinated"].mean(),4) if len(sg) else None
            mit_rows.append({"model":m,"precision":p,"dataset":d,"chain_type":ct,"step5_H_rate":hr})
    ws5 = wb.create_sheet("Mitigation"); _ws(ws5, pd.DataFrame(mit_rows))

    wb.save(RESULTS_XLSX)
    total = len(df)
    print(f"Saved {RESULTS_XLSX} — {total} rows across 5 sheets.")

def _ws(ws, df):
    from openpyxl.styles import Font, PatternFill, Alignment
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(row=1, column=ci, value=col)
        c.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        c.fill = PatternFill("solid", start_color="2E75B6")
        c.alignment = Alignment(horizontal="center")
    for ri, row in enumerate(df.itertuples(index=False), 2):
        for ci, val in enumerate(row, 1):
            ws.cell(row=ri, column=ci, value=val)
    for col in ws.columns:
        w = max(len(str(c.value or "")) for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(w+4, 45)

# ── ENTRY POINT ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("="*60)
    print(f"SnowBallScan v10 |  SAMPLE_SIZE={SAMPLE_SIZE}  |  K={CHAIN_LENGTH}  |  FP16 + robust CSV loading")
    print(f"Models:     {list(MODELS.keys())}")
    print(f"Precisions: {PRECISIONS}")
    print(f"Datasets:   {DATASETS}")
    print(f"Checkpoint: {CHECKPOINT}")
    print(f"Output:     {RESULTS_XLSX}")
    print("="*60+"\n")
    run_experiment()

In [ ]:
import json, shutil

# Restore checkpoint from dataset
src = "/kaggle/input/datasets/samuelstephen77/snowballscan-dataset/checkpoint.json"
dst = "/kaggle/working/checkpoint_v10.json"
shutil.copy(src, dst)

with open(dst) as f:
    ckpt = json.load(f)
print(f"Restored: {len(ckpt['done_keys'])} chains done")

In [ ]:
pip install rank-bm25

In [ ]:
import os, shutil
from pathlib import Path

working = Path("/kaggle/working")

for item in working.iterdir():
    if item.name == ".virtual_documents":
        continue  # keep kaggle system folder
    try:
        if item.is_file():
            item.unlink()
            print(f"Deleted file: {item.name}")
        elif item.is_dir():
            shutil.rmtree(item)
            print(f"Deleted folder: {item.name}")
    except Exception as e:
        print(f"Could not delete {item.name}: {e}")

print("\nDone. Remaining:", [f.name for f in working.iterdir()])

In [ ]:
# ============================================================
# SnowBallScan — Extra Experiments
# Run 1: Neutral Context (no corrective framing) — HaluEval, n=200, K=5
# Run 2: K=10 Chains — all datasets, n=100, K=10
# ============================================================
# USAGE:
#   Change RUN_MODE below then run the notebook
#   RUN_MODE = "neutral"  → Run 1 (~1 hour)
#   RUN_MODE = "k10"      → Run 2 (~2.5 hours)
#
# Each run has its own checkpoint and results file.
# Restore checkpoint from dataset if session crashes:
#   import shutil
#   shutil.copy("/kaggle/input/datasets/samuelstephen77/snowballscan-dataset/checkpoint_neutral.json",
#               "/kaggle/working/checkpoint_neutral.json")
# ============================================================

import os, json, random, warnings, re, sys
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
RUN_MODE = "k10"   # ← Change to "k10" for second run

if RUN_MODE == "neutral":
    SAMPLE_SIZE   = 200
    CHAIN_LENGTH  = 5
    DATASETS      = ["halueval"]        # HaluEval only
    CHECKPOINT    = "checkpoint_neutral.json"
    RESULTS_XLSX  = "results_neutral.xlsx"
    DESCRIPTION   = "Neutral context — no corrective framing — HaluEval n=200 K=5"

elif RUN_MODE == "k10":
    SAMPLE_SIZE   = 100
    CHAIN_LENGTH  = 10
    DATASETS      = ["truthfulqa", "freshqa", "halueval"]
    CHECKPOINT    = "checkpoint_k10.json"
    RESULTS_XLSX  = "results_k10.xlsx"
    DESCRIPTION   = "K=10 chains — all datasets n=100 K=10"

else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}. Use 'neutral' or 'k10'.")

PAI_THRESHOLD = 1.5
SEED          = 42
BASE          = "/kaggle/input/datasets/samuelstephen77/snowballscan-dataset"

MODELS = {
    "phi2":      "microsoft/phi-2",
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "qwen25":    "Qwen/Qwen2.5-1.5B-Instruct",
}
PRECISIONS = ["fp16"]

random.seed(SEED)
np.random.seed(SEED)

# ── INSTALL ───────────────────────────────────────────────────────────────────
print("Installing dependencies...")
os.system("pip install -q rank_bm25 sentence-transformers datasets openpyxl")
print("Ready.\n")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

# ── PROMPT TEMPLATES ─────────────────────────────────────────────────────────
def format_prompt(model_name, question, prior_response=None, neutral=False):
    """
    neutral=True: pass prior response as raw context with NO corrective framing
    neutral=False: standard prompt with "answer more carefully" framing
    """
    if prior_response and not neutral:
        ctx = prior_response[:120].strip()
        context_hint = f"Your previous answer was: \"{ctx}\". Now answer again more carefully."
    elif prior_response and neutral:
        # Neutral — just show prior response as context, no correction instruction
        ctx = prior_response[:120].strip()
        context_hint = f"Context: \"{ctx}\"."
    else:
        context_hint = None

    if model_name == "tinyllama":
        sys_msg = "You are a factual assistant. Answer questions with a single sentence."
        user_msg = f"{context_hint}\n\nQuestion: {question}\nAnswer:" if context_hint else f"Question: {question}\nAnswer:"
        return f"<|system|>\n{sys_msg}</s>\n<|user|>\n{user_msg}</s>\n<|assistant|>\n"

    elif model_name == "phi2":
        if context_hint:
            return f"Instruct: {context_hint} Answer the following question in one sentence.\nQuestion: {question}\nOutput:"
        return f"Instruct: Answer the following question in one sentence.\nQuestion: {question}\nOutput:"

    elif model_name == "qwen25":
        sys_msg = "You are a factual assistant. Answer each question with one clear, concise sentence."
        user_msg = f"{context_hint}\n\nQuestion: {question}" if context_hint else question
        return (f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n")
    else:
        if context_hint:
            return f"{context_hint}\nQuestion: {question}\nAnswer in one sentence:"
        return f"Question: {question}\nAnswer in one sentence:"

# ── CHECKPOINT ────────────────────────────────────────────────────────────────
def load_checkpoint():
    if Path(CHECKPOINT).exists():
        with open(CHECKPOINT) as f:
            data = json.load(f)
        print(f"Resuming: {len(data['rows'])} rows, {len(data['done_keys'])} chains done.")
        return data
    print("No checkpoint — starting fresh.")
    return {"rows": [], "done_keys": []}

def save_checkpoint(data):
    with open(CHECKPOINT, "w") as f:
        json.dump(data, f, indent=2)

def make_key(model_name, precision, dataset, q_idx):
    return f"{model_name}|{precision}|{dataset}|{q_idx}"

# ── DATASET LOADERS ───────────────────────────────────────────────────────────
def load_datasets_all(sample_size, datasets_to_load):
    out = {}
    import pandas as _pd, json as _json, random as _r

    if "truthfulqa" in datasets_to_load:
        print("Loading TruthfulQA...")
        tqa_path = f"{BASE}/TruthfulQA.csv"
        if Path(tqa_path).exists():
            df = _pd.read_csv(tqa_path)
            df.columns = [c.strip() for c in df.columns]
            q_col = next(c for c in df.columns if 'question' in c.lower())
            a_col = next(c for c in df.columns if 'best' in c.lower() and 'answer' in c.lower())
            cor_col = next((c for c in df.columns if 'correct' in c.lower() and 'answer' in c.lower()), None)
            df = df[df[q_col].notna()].sample(min(sample_size, len(df)), random_state=SEED)
            items = []
            for _, row in df.iterrows():
                q = str(row[q_col]).strip()
                gold = str(row[a_col]).strip()
                corpus = [a.strip() for a in str(row[cor_col]).split(';') if a.strip() and a.strip() != 'nan'] if cor_col else [gold]
                if q and gold:
                    items.append({"question": q, "gold": gold, "corpus": corpus or [gold]})
            out["truthfulqa"] = items
            print(f"  TruthfulQA: {len(items)} ✓")
        else:
            tqa = load_dataset("truthful_qa", "generation", split="validation")
            tqa = tqa.shuffle(seed=SEED).select(range(min(sample_size, len(tqa))))
            out["truthfulqa"] = [{"question": r["question"], "gold": r["best_answer"],
                                   "corpus": r["correct_answers"] or [r["best_answer"]]} for r in tqa]
            print(f"  TruthfulQA HF: {len(out['truthfulqa'])} ✓")

    if "freshqa" in datasets_to_load:
        print("Loading FreshQA...")
        fqa_path = f"{BASE}/freshqa.csv"
        if Path(fqa_path).exists():
            df = _pd.read_csv(fqa_path, skiprows=2, header=None)
            df.columns = ['id','split','question','effective_year','next_review','false_premise',
                          'num_hops','fact_type','source','answer_0','answer_1','answer_2',
                          'answer_3','answer_4','answer_5','answer_6','answer_7','answer_8','answer_9','note']
            df = df[df['question'].notna()]
            df = df[df['false_premise'].astype(str).str.upper() != 'TRUE']
            df = df[df['question'] != 'question'].reset_index(drop=True)
            df = df.sample(min(sample_size, len(df)), random_state=SEED)
            items = []
            for _, row in df.iterrows():
                q = str(row['question']).strip()
                answers = [str(row[f'answer_{i}']).strip() for i in range(10)
                           if str(row.get(f'answer_{i}','')).strip() not in ['','nan','NaN']]
                if q and answers:
                    items.append({"question": q, "gold": answers[0], "corpus": answers})
            out["freshqa"] = items
            print(f"  FreshQA: {len(items)} ✓")
        else:
            print("  FreshQA CSV not found — skipping")

    if "halueval" in datasets_to_load:
        print("Loading HaluEval...")
        hqa_path = f"{BASE}/halueval.json"
        loaded = False
        if Path(hqa_path).exists():
            try:
                with open(hqa_path) as f:
                    raw = []
                    for line in f:
                        line = line.strip()
                        if line:
                            try: raw.append(json.loads(line))
                            except: pass
                _r.seed(SEED); _r.shuffle(raw)
                items = []
                for r in raw[:sample_size]:
                    q = str(r.get("question","")).strip()
                    gold = str(r.get("right_answer","")).strip()
                    hall = str(r.get("hallucinated_answer","")).strip()
                    if q and gold:
                        items.append({"question":q,"gold":gold,
                                      "hallucinated":hall if hall else None,"corpus":[gold]})
                if items:
                    out["halueval"] = items
                    print(f"  HaluEval JSON: {len(items)} ✓")
                    loaded = True
            except Exception as e:
                print(f"  JSON failed: {e}")
        if not loaded:
            hqa = load_dataset("pminervini/HaluEval", "qa", split="data")
            hqa = hqa.shuffle(seed=SEED).select(range(min(sample_size, len(hqa))))
            out["halueval"] = [{"question":r["question"],"gold":r["right_answer"],
                                 "hallucinated":r["hallucinated_answer"],"corpus":[r["right_answer"]]} for r in hqa]
            print(f"  HaluEval HF: {len(out['halueval'])} ✓")

    print(f"\nDatasets ready: { {k:len(v) for k,v in out.items()} }\n")
    return out

# ── MODEL LOADER ──────────────────────────────────────────────────────────────
def load_model(model_name, model_id, precision):
    print(f"Loading {model_name} @ {precision}...")
    torch.manual_seed(SEED)
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)
    if not hasattr(config,'pad_token_id') or config.pad_token_id is None:
        config.pad_token_id = 2
    model = AutoModelForCausalLM.from_pretrained(
        model_id, config=config, dtype=torch.float16,
        device_map="auto", trust_remote_code=True)
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
        tok.pad_token_id = tok.eos_token_id
    if not hasattr(model.config,'pad_token_id') or model.config.pad_token_id is None:
        model.config.pad_token_id = tok.pad_token_id
    model.eval()
    mem = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
    print(f"  {model_name} ready. GPU: {mem:.2f}GB\n")
    return model, tok

# ── REPETITION DETECTOR ───────────────────────────────────────────────────────
def is_repetition_loop(text):
    if not text or len(text.strip()) < 5:
        return True
    words = text.lower().split()
    if len(words) > 10:
        from collections import Counter
        trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
        if Counter(trigrams).most_common(1)[0][1] >= 4:
            return True
    return False

# ── GENERATION ────────────────────────────────────────────────────────────────
def generate_response(model, tok, model_name, question, prior_response=None,
                      max_new_tokens=100, neutral=False):
    prompt = format_prompt(model_name, question, prior_response, neutral=neutral)
    inputs = tok(prompt, return_tensors="pt", truncation=True,
                 max_length=768, padding=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, temperature=1.0,
                             repetition_penalty=1.3,
                             pad_token_id=tok.pad_token_id,
                             eos_token_id=tok.eos_token_id)
    new = out[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(new, skip_special_tokens=True).strip()
    for sep in [". ", ".\n", "\n"]:
        if sep in text:
            text = text.split(sep)[0].strip() + "."
            break
    if is_repetition_loop(text):
        inputs2 = tok(format_prompt(model_name, question, None),
                      return_tensors="pt", truncation=True,
                      max_length=512, padding=False).to(model.device)
        with torch.no_grad():
            out2 = model.generate(**inputs2, max_new_tokens=80,
                                  do_sample=False, temperature=1.0,
                                  repetition_penalty=1.8,
                                  pad_token_id=tok.pad_token_id,
                                  eos_token_id=tok.eos_token_id)
        new2 = out2[0][inputs2["input_ids"].shape[1]:]
        text2 = tok.decode(new2, skip_special_tokens=True).strip()
        for sep in [". ", ".\n", "\n"]:
            if sep in text2:
                text2 = text2.split(sep)[0].strip() + "."
                break
        if not is_repetition_loop(text2):
            text = text2
    return text if text else "No answer generated."

# ── HALLUCINATION DETECTION ───────────────────────────────────────────────────
def load_entailment_model():
    print("Loading DeBERTa-v3-base cross-encoder...")
    m = CrossEncoder("cross-encoder/nli-deberta-v3-base")
    print("  Ready.\n")
    return m

def is_hallucinated(entail_model, response, gold):
    if not response or not gold:
        return 1
    try:
        scores = entail_model.predict([(gold, response)])
        e_score = float(scores[0][2]) if scores[0].shape[0]==3 else float(scores[0][-1])
    except:
        e_score = 0.0
    gold_tok = set(gold.lower().split())
    resp_tok = set(response.lower().split())
    overlap = len(gold_tok & resp_tok) / max(len(gold_tok), 1)
    return 0 if (e_score > 0.4 or overlap > 0.3) else 1

# ── BM25 ──────────────────────────────────────────────────────────────────────
def build_bm25(corpus):
    tok = [d.lower().split() for d in corpus if d]
    return BM25Okapi(tok) if tok else None

def retrieve_correction(bm25, corpus, query):
    if not bm25 or not corpus:
        return corpus[0] if corpus else ""
    return corpus[int(np.argmax(bm25.get_scores(query.lower().split())))]

# ── CHAIN RUNNER ─────────────────────────────────────────────────────────────
def run_chain(model, tok, model_name, entail_model,
              question, gold, corpus,
              seeded_response=None, apply_mitigation=False,
              non_propagating=False, neutral_context=False):
    """
    neutral_context=True: pass prior response without corrective framing
    neutral_context=False: standard corrective framing ("answer more carefully")
    """
    bm25 = build_bm25(corpus)
    last_response = None
    results = []

    for k in range(1, CHAIN_LENGTH + 1):
        ctx = None if non_propagating else last_response

        if k == 1 and seeded_response is not None:
            resp = seeded_response
        else:
            resp = generate_response(model, tok, model_name, question,
                                     prior_response=ctx,
                                     neutral=neutral_context)

        h = is_hallucinated(entail_model, resp, gold)

        if apply_mitigation and k in [2, 4] and h == 1:
            correction = retrieve_correction(bm25, corpus, question)
            if correction:
                resp = correction
                h = 0

        results.append({
            "step": k, "response": resp[:300],
            "hallucinated": h,
            "loop_detected": int(is_repetition_loop(resp)),
        })

        if not non_propagating:
            last_response = resp

    return results

# ── PAI & ONSET ───────────────────────────────────────────────────────────────
def compute_pai(h_rates):
    if not h_rates or h_rates[0] == 0:
        return [1.0 if h == 0 else 2.0 for h in h_rates]
    return [round(h / h_rates[0], 4) for h in h_rates]

def compute_onset(pai_vals, threshold=PAI_THRESHOLD):
    for i, v in enumerate(pai_vals):
        if v >= threshold:
            return i + 1
    return None

# ── MAIN EXPERIMENT ───────────────────────────────────────────────────────────
def run_experiment():
    ckpt = load_checkpoint()
    done_keys = set(ckpt["done_keys"])
    rows = ckpt["rows"]

    datasets = load_datasets_all(SAMPLE_SIZE, DATASETS)
    entail_model = load_entailment_model()

    # Determine neutral flag
    is_neutral = (RUN_MODE == "neutral")

    for model_name, model_id in MODELS.items():
        try:
            model, tok = load_model(model_name, model_id, "fp16")
        except Exception as e:
            print(f"SKIP {model_name}: {e}\n")
            continue

        for dataset_name in DATASETS:
            questions = datasets.get(dataset_name, [])
            if not questions:
                continue

            sf    = {k: [] for k in range(1, CHAIN_LENGTH+1)}
            sf_np = {k: [] for k in range(1, CHAIN_LENGTH+1)}

            for q_idx, qdata in enumerate(questions):
                key = make_key(model_name, "fp16", dataset_name, q_idx)
                if key in done_keys:
                    print(f"  SKIP: {key}")
                    continue

                print(f"  [{model_name}|{dataset_name}|q{q_idx}] Q: {qdata['question'][:60]}...")

                q         = qdata["question"]
                gold      = qdata["gold"]
                corpus    = qdata["corpus"]
                seed_resp = qdata.get("hallucinated", None)

                try:
                    # Main chain
                    chain = run_chain(model, tok, model_name, entail_model,
                                      q, gold, corpus,
                                      seeded_response=seed_resp,
                                      neutral_context=is_neutral)

                    # Non-propagating baseline
                    chain_np = run_chain(model, tok, model_name, entail_model,
                                         q, gold, corpus,
                                         seeded_response=None,
                                         non_propagating=True,
                                         neutral_context=False)

                    for s in chain:
                        sf[s["step"]].append(s["hallucinated"])
                    for s in chain_np:
                        sf_np[s["step"]].append(s["hallucinated"])

                    chain_type = "neutral_propagating" if is_neutral else "propagating"
                    for s in chain:
                        rows.append({
                            "model": model_name, "precision": "fp16",
                            "dataset": dataset_name, "q_idx": q_idx,
                            "chain_type": chain_type,
                            "step": s["step"], "hallucinated": s["hallucinated"],
                            "loop_detected": s["loop_detected"],
                            "response": s["response"],
                            "run_mode": RUN_MODE,
                        })
                    for s in chain_np:
                        rows.append({
                            "model": model_name, "precision": "fp16",
                            "dataset": dataset_name, "q_idx": q_idx,
                            "chain_type": "non_propagating",
                            "step": s["step"], "hallucinated": s["hallucinated"],
                            "loop_detected": s["loop_detected"],
                            "response": s["response"],
                            "run_mode": RUN_MODE,
                        })

                    done_keys.add(key)
                    ckpt["rows"] = rows
                    ckpt["done_keys"] = list(done_keys)
                    save_checkpoint(ckpt)

                    flags = [s["hallucinated"] for s in chain]
                    print(f"    H-flags: {flags}  Loops: {sum(s['loop_detected'] for s in chain)}/5")

                except Exception as e:
                    print(f"    ERROR: {e}")
                    import traceback; traceback.print_exc()
                    continue

            # PAI summary
            h  = [np.mean(sf[k])    if sf[k]    else 0.0 for k in range(1, CHAIN_LENGTH+1)]
            hn = [np.mean(sf_np[k]) if sf_np[k] else 0.0 for k in range(1, CHAIN_LENGTH+1)]
            pai  = compute_pai(h)
            pain = compute_pai(hn)
            onset = compute_onset(pai)

            print(f"\n  ── [{model_name}|{dataset_name}] ──")
            for k in range(CHAIN_LENGTH):
                print(f"     Step {k+1}: H={h[k]:.3f}  PAI={pai[k]:.3f}")
            print(f"  PAI@{CHAIN_LENGTH}: {pai[-1]:.3f} | Non-prop: {pain[-1]:.3f}")
            print(f"  Onset: {'Step '+str(onset) if onset else 'Not Reached (∞)'}\n")

        del model, tok
        torch.cuda.empty_cache()
        print(f"GPU cleared: {model_name}\n")

    print("All chains done. Building XLSX...")
    build_results_xlsx(rows)

# ── RESULTS XLSX ──────────────────────────────────────────────────────────────
def build_results_xlsx(rows):
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment

    if not rows:
        print("No rows.")
        return

    df = pd.DataFrame(rows)
    wb = openpyxl.Workbook()

    # Sheet 1: Raw
    ws1 = wb.active; ws1.title = "Raw Chains"
    _ws(ws1, df)

    # Sheet 2: PAI by step
    chain_type_filter = "neutral_propagating" if RUN_MODE == "neutral" else "propagating"
    prop = df[df["chain_type"] == chain_type_filter]
    summ = []
    for (m, d), g in prop.groupby(["model", "dataset"]):
        h1 = g[g["step"]==1]["hallucinated"].mean() if len(g[g["step"]==1]) else 0
        for step in range(1, CHAIN_LENGTH+1):
            sg = g[g["step"]==step]
            hr = sg["hallucinated"].mean() if len(sg) else 0.0
            pai = round(hr/h1, 4) if h1 > 0 else 1.0
            summ.append({"model":m,"dataset":d,"step":step,
                         "H_rate":round(hr,4),"PAI":pai,"n":len(sg)})
    ws2 = wb.create_sheet("PAI by Step"); _ws(ws2, pd.DataFrame(summ))

    # Sheet 3: PAI@final step
    ds = pd.DataFrame(summ)
    ws3 = wb.create_sheet(f"PAI at Step {CHAIN_LENGTH}")
    _ws(ws3, ds[ds["step"]==CHAIN_LENGTH][
        ["model","dataset","H_rate","PAI","n"]].reset_index(drop=True))

    # Sheet 4: Comparison vs non-prop
    comp = []
    for (m, d), g in df.groupby(["model","dataset"]):
        for ct in [chain_type_filter, "non_propagating"]:
            sg = g[(g["chain_type"]==ct) & (g["step"]==CHAIN_LENGTH)]
            hr = round(sg["hallucinated"].mean(), 4) if len(sg) else None
            comp.append({"model":m,"dataset":d,"chain_type":ct,
                         f"step{CHAIN_LENGTH}_H_rate":hr})
    ws4 = wb.create_sheet("Comparison"); _ws(ws4, pd.DataFrame(comp))

    wb.save(RESULTS_XLSX)
    print(f"\nSaved {RESULTS_XLSX} ({len(df)} rows, {wb.sheetnames})")

def _ws(ws, df):
    from openpyxl.styles import Font, PatternFill, Alignment
    for ci, col in enumerate(df.columns, 1):
        c = ws.cell(row=1, column=ci, value=col)
        c.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        c.fill = PatternFill("solid", start_color="2E75B6")
        c.alignment = Alignment(horizontal="center")
    for ri, row in enumerate(df.itertuples(index=False), 2):
        for ci, val in enumerate(row, 1):
            ws.cell(row=ri, column=ci, value=val)
    for col in ws.columns:
        w = max(len(str(c.value or "")) for c in col)
        ws.column_dimensions[col[0].column_letter].width = min(w+4, 45)

# ── ENTRY POINT ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("="*60)
    print(f"RUN_MODE:    {RUN_MODE}")
    print(f"DESCRIPTION: {DESCRIPTION}")
    print(f"SAMPLE_SIZE: {SAMPLE_SIZE}")
    print(f"CHAIN_LENGTH:{CHAIN_LENGTH}")
    print(f"DATASETS:    {DATASETS}")
    print(f"CHECKPOINT:  {CHECKPOINT}")
    print(f"OUTPUT:      {RESULTS_XLSX}")
    print("="*60+"\n")
    run_experiment()

In [ ]:
# ============================================================
# SnowBallScan v2 — Extended Experiments (FIXED)
# Run A: New architectures (Gemma-2B + Llama 3.2 3B)
# Run B: Temperature sensitivity (0.0, 0.3, 0.7)
#
# FIXES:
#   - Fixed syntax error with global BASE
#   - Fixed dataset paths (correct Kaggle mount point)
#   - Added retry logic for slow downloads
#   - Better error handling
#   - Progress bars for downloads
#   - Fixed authentication issues
#   - Added timeout protection
# ============================================================

import os, json, random, warnings, re, time, sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ── HF LOGIN ──────────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login as hf_login

print("Authenticating...")
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
    hf_login(token=hf_token, add_to_git_credential=False)
    print("✅ HF login OK")
else:
    print("⚠️  No HF token found. Some models may not load.")

# ── CONFIG ────────────────────────────────────────────────────────────────────
RUN_MODE = "A"   # ← Change to "B" for temperature sensitivity run

# Try multiple possible dataset paths
POSSIBLE_PATHS = [
    "/kaggle/input/snowballscan-dataset",
    "/kaggle/input/datasets/kevinsam77/snowballscan-dataset",
    "/kaggle/input/datasets/samuelstephen77/snowballscan-dataset",
    "/kaggle/input/datasets/",
    "/kaggle/input/",
]

# Find the correct BASE path
BASE = None
for path in POSSIBLE_PATHS:
    if Path(path).exists():
        # Check if it contains dataset files
        files = list(Path(path).glob("*.csv")) + list(Path(path).glob("*.json"))
        if files:
            BASE = path
            print(f"✅ Found dataset path: {BASE}")
            print(f"   Files: {[f.name for f in files[:5]]}")
            break

if BASE is None:
    print("⚠️  No dataset path found! Will try to load from HuggingFace.")
    BASE = "/kaggle/input/"  # fallback

# Run A settings
RUN_A_MODELS = {
    "gemma2b":    "google/gemma-2b",
    "llama32_3b": "meta-llama/Llama-3.2-3B",
}
RUN_A_DATASETS    = ["truthfulqa", "freshqa", "halueval"]
RUN_A_SAMPLE_SIZE = 200
RUN_A_CHAIN_LEN   = 5
RUN_A_TEMPS       = [0.0]  # greedy only for Run A

# Run B settings
RUN_B_MODELS = {
    "phi2":      "microsoft/phi-2",
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "qwen25":    "Qwen/Qwen2.5-1.5B-Instruct",
}
RUN_B_DATASETS    = ["truthfulqa", "halueval"]
RUN_B_SAMPLE_SIZE = 50
RUN_B_CHAIN_LEN   = 5
RUN_B_TEMPS       = [0.0, 0.3, 0.7]

# Shared
SEED               = 42
PAI_THRESHOLD      = 1.5
MASTER_CHECKPOINT  = "snowballscan_v2_master.json"
MASTER_RESULTS     = "snowballscan_v2_master.xlsx"
CHECKPOINT_EVERY   = 10

random.seed(SEED)
np.random.seed(SEED)

# ── INSTALL ───────────────────────────────────────────────────────────────────
print("\nInstalling dependencies...")
os.system("pip install -q rank_bm25 sentence-transformers datasets openpyxl tqdm")
print("✅ Dependencies ready\n")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

# ── MASTER CHECKPOINT ─────────────────────────────────────────────────────────
def load_master_checkpoint():
    if Path(MASTER_CHECKPOINT).exists():
        with open(MASTER_CHECKPOINT) as f:
            data = json.load(f)
        print(f"✅ Resumed: {len(data['done_keys'])} chains done, {len(data['rows'])} rows")
        return data
    print("Fresh start — no checkpoint found")
    return {"rows": [], "done_keys": [], "run_modes_done": []}

def save_master_checkpoint(data):
    with open(MASTER_CHECKPOINT, "w") as f:
        json.dump(data, f, indent=2)

def make_key(run_mode, model_name, dataset, q_idx, temp=0.0):
    return f"{run_mode}|{model_name}|{dataset}|{q_idx}|t{temp}"

# ── PROMPT TEMPLATES ──────────────────────────────────────────────────────────
def format_prompt(model_name, question, prior_response=None):
    if prior_response:
        ctx = prior_response[:120].strip()
        context_hint = f"Your previous answer was: \"{ctx}\". Now answer again more carefully."
    else:
        context_hint = None

    if model_name in ["tinyllama"]:
        sys_msg = "You are a factual assistant. Answer questions with a single sentence."
        user_msg = (f"{context_hint}\n\nQuestion: {question}\nAnswer:"
                    if context_hint else f"Question: {question}\nAnswer:")
        return f"<|system|>\n{sys_msg}</s>\n<|user|>\n{user_msg}</s>\n<|assistant|>\n"

    elif model_name == "phi2":
        if context_hint:
            return (f"Instruct: {context_hint} Answer the following question in one sentence.\n"
                    f"Question: {question}\nOutput:")
        return f"Instruct: Answer the following question in one sentence.\nQuestion: {question}\nOutput:"

    elif model_name == "qwen25":
        sys_msg = "You are a factual assistant. Answer each question with one clear, concise sentence."
        user_msg = f"{context_hint}\n\nQuestion: {question}" if context_hint else question
        return (f"<|im_start|>system\n{sys_msg}<|im_end|>\n"
                f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                f"<|im_start|>assistant\n")

    elif model_name in ["gemma2b"]:
        if context_hint:
            return f"<start_of_turn>user\n{context_hint}\n\nQuestion: {question}<end_of_turn>\n<start_of_turn>model\n"
        return f"<start_of_turn>user\nQuestion: {question}<end_of_turn>\n<start_of_turn>model\n"

    elif model_name in ["llama32_3b"]:
        sys = "You are a factual assistant. Answer each question with one clear, concise sentence."
        if context_hint:
            usr = f"{context_hint}\n\nQuestion: {question}"
        else:
            usr = f"Question: {question}"
        return (f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{sys}<|eot_id|>"
                f"<|start_header_id|>user<|end_header_id|>\n{usr}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n")

    else:
        if context_hint:
            return f"{context_hint}\nQuestion: {question}\nAnswer in one sentence:"
        return f"Question: {question}\nAnswer in one sentence:"

# ── REPETITION DETECTOR ───────────────────────────────────────────────────────
def is_repetition_loop(text):
    if not text or len(text.strip()) < 5:
        return True
    words = text.lower().split()
    if len(words) > 10:
        from collections import Counter
        trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
        if Counter(trigrams).most_common(1)[0][1] >= 4:
            return True
    return False

# ── MODEL LOADER ──────────────────────────────────────────────────────────────
def load_model(model_name, model_id):
    print(f"\nLoading {model_name} ({model_id})...")
    torch.manual_seed(SEED)

    config = AutoConfig.from_pretrained(
        model_id, trust_remote_code=True, token=hf_token if hf_token else None)
    if not getattr(config, 'pad_token_id', None):
        config.pad_token_id = 2

    model = AutoModelForCausalLM.from_pretrained(
        model_id, config=config,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        token=hf_token if hf_token else None
    )
    tok = AutoTokenizer.from_pretrained(
        model_id, trust_remote_code=True, token=hf_token if hf_token else None)
    
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
        tok.pad_token_id = tok.eos_token_id
    if not getattr(model.config, 'pad_token_id', None):
        model.config.pad_token_id = tok.pad_token_id

    model.eval()
    mem = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    print(f"  ✅ {model_name} ready | GPU: {mem:.2f}GB")
    return model, tok

# ── GENERATION ────────────────────────────────────────────────────────────────
def generate_response(model, tok, model_name, question,
                      prior_response=None, temperature=0.0, max_new_tokens=100):
    prompt = format_prompt(model_name, question, prior_response)
    inputs = tok(prompt, return_tensors="pt", truncation=True,
                 max_length=768, padding=False).to(model.device)

    do_sample = temperature > 0.0
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            repetition_penalty=1.3,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(new_tokens, skip_special_tokens=True).strip()

    # Truncate to first sentence
    for sep in [". ", ".\n", "\n"]:
        if sep in text:
            text = text.split(sep)[0].strip() + "."
            break

    # Retry if loop detected
    if is_repetition_loop(text):
        prompt2 = format_prompt(model_name, question, None)
        inputs2 = tok(prompt2, return_tensors="pt", truncation=True,
                      max_length=512, padding=False).to(model.device)
        with torch.no_grad():
            out2 = model.generate(**inputs2, max_new_tokens=80,
                                  do_sample=False, temperature=1.0,
                                  repetition_penalty=1.8,
                                  pad_token_id=tok.pad_token_id,
                                  eos_token_id=tok.eos_token_id)
        new2 = out2[0][inputs2["input_ids"].shape[1]:]
        text2 = tok.decode(new2, skip_special_tokens=True).strip()
        for sep in [". ", ".\n", "\n"]:
            if sep in text2:
                text2 = text2.split(sep)[0].strip() + "."
                break
        if not is_repetition_loop(text2):
            text = text2

    return text if text else "No answer generated."

# ── HALLUCINATION DETECTOR ────────────────────────────────────────────────────
def load_detector():
    print("Loading DeBERTa detector...")
    try:
        # Set timeout for download
        os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"  # 5 minutes
        
        m = CrossEncoder(
            "cross-encoder/nli-deberta-v3-base",
            trust_remote_code=True,
            device="cuda" if torch.cuda.is_available() else "cpu"
        )
        print("  ✅ Detector ready")
        return m
    except Exception as e:
        print(f"  ⚠️  Failed to load DeBERTa: {e}")
        print("  Trying fallback (MiniLM)...")
        try:
            m = CrossEncoder("cross-encoder/nli-MiniLM2-L6-H768")
            print("  ✅ Using MiniLM detector (smaller, faster)")
            return m
        except:
            print("  ❌ Cannot load any detector. Hallucination detection will be limited.")
            return None

def is_hallucinated(detector, response, gold):
    if detector is None or not response or not gold:
        return 1
    try:
        scores = detector.predict([(gold, response)])
        e_score = float(scores[0][2]) if scores[0].shape[0] == 3 else float(scores[0][-1])
    except:
        e_score = 0.0
    gold_tok = set(gold.lower().split())
    resp_tok = set(response.lower().split())
    overlap = len(gold_tok & resp_tok) / max(len(gold_tok), 1)
    return 0 if (e_score > 0.4 or overlap > 0.3) else 1

# ── DATASET LOADERS ───────────────────────────────────────────────────────────
# ── DATASET LOADERS ───────────────────────────────────────────────────────────
def load_datasets(sample_size, datasets_to_load):
    import pandas as _pd, json as _json, random as _r
    out = {}
    
    # Use the global BASE path
    base_path = BASE
    print(f"Looking for datasets in: {base_path}")
    
    # List available files
    if Path(base_path).exists():
        all_files = list(Path(base_path).glob("*"))
        csv_files = [f for f in all_files if f.suffix in ['.csv', '.CSV']]
        json_files = [f for f in all_files if f.suffix in ['.json', '.JSON']]
        print(f"Found {len(csv_files)} CSV files and {len(json_files)} JSON files")
        if csv_files:
            print(f"  CSV files: {[f.name for f in csv_files]}")
        if json_files:
            print(f"  JSON files: {[f.name for f in json_files]}")
    else:
        print(f"⚠️  Path {base_path} does not exist")
        print("Will try to load datasets from HuggingFace instead")

    if "truthfulqa" in datasets_to_load:
        print("\nLoading TruthfulQA...")
        tqa_path = f"{base_path}/TruthfulQA.csv"
        if Path(tqa_path).exists():
            try:
                df = _pd.read_csv(tqa_path)
                df.columns = [c.strip() for c in df.columns]
                q_col = next(c for c in df.columns if 'question' in c.lower())
                a_col = next(c for c in df.columns if 'best' in c.lower() and 'answer' in c.lower())
                cor_col = next((c for c in df.columns if 'correct' in c.lower() and 'answer' in c.lower()), None)
                df = df[df[q_col].notna()].sample(min(sample_size, len(df)), random_state=SEED)
                items = []
                for _, row in df.iterrows():
                    q = str(row[q_col]).strip()
                    gold = str(row[a_col]).strip()
                    corpus = [a.strip() for a in str(row[cor_col]).split(';')
                              if a.strip() and a.strip() != 'nan'] if cor_col else [gold]
                    if q and gold:
                        items.append({"question": q, "gold": gold, "corpus": corpus or [gold]})
                out["truthfulqa"] = items
                print(f"  ✅ TruthfulQA: {len(items)} samples from CSV")
            except Exception as e:
                print(f"  ⚠️  TruthfulQA CSV loading failed: {e}")
                print("  Falling back to HuggingFace...")
                try:
                    ds = load_dataset("truthful_qa", "generation", split="validation")
                    ds = ds.shuffle(seed=SEED).select(range(min(sample_size, len(ds))))
                    out["truthfulqa"] = [{"question": r["question"], "gold": r["best_answer"],
                                           "corpus": r["correct_answers"] or [r["best_answer"]]} for r in ds]
                    print(f"  ✅ TruthfulQA HF: {len(out['truthfulqa'])} samples")
                except Exception as e2:
                    print(f"  ❌ Failed to load TruthfulQA: {e2}")
        else:
            print("  TruthfulQA CSV not found, loading from HuggingFace...")
            try:
                ds = load_dataset("truthful_qa", "generation", split="validation")
                ds = ds.shuffle(seed=SEED).select(range(min(sample_size, len(ds))))
                out["truthfulqa"] = [{"question": r["question"], "gold": r["best_answer"],
                                       "corpus": r["correct_answers"] or [r["best_answer"]]} for r in ds]
                print(f"  ✅ TruthfulQA HF: {len(out['truthfulqa'])} samples")
            except Exception as e:
                print(f"  ❌ Failed to load TruthfulQA: {e}")

    if "freshqa" in datasets_to_load:
        print("\nLoading FreshQA...")
        fqa_path = f"{base_path}/freshqa.csv"
        if Path(fqa_path).exists():
            try:
                df = _pd.read_csv(fqa_path, skiprows=2, header=None)
                # Handle variable column count
                num_cols = len(df.columns)
                base_cols = ['id','split','question','effective_year','next_review','false_premise',
                            'num_hops','fact_type','source']
                answer_cols = [f'answer_{i}' for i in range(10)]
                df.columns = base_cols + answer_cols[:num_cols - len(base_cols)] + ['note']
                df = df[df['question'].notna()]
                df = df[df['false_premise'].astype(str).str.upper() != 'TRUE']
                df = df[df['question'] != 'question'].reset_index(drop=True)
                df = df.sample(min(sample_size, len(df)), random_state=SEED)
                items = []
                for _, row in df.iterrows():
                    q = str(row['question']).strip()
                    answers = [str(row[f'answer_{i}']).strip() for i in range(10)
                               if str(row.get(f'answer_{i}', '')).strip() not in ['', 'nan', 'NaN']]
                    if q and answers:
                        items.append({"question": q, "gold": answers[0], "corpus": answers})
                out["freshqa"] = items
                print(f"  ✅ FreshQA: {len(items)} samples from CSV")
            except Exception as e:
                print(f"  ⚠️  FreshQA loading failed: {e}")
        else:
            print("  FreshQA CSV not found — skipping")

    if "halueval" in datasets_to_load:
        print("\nLoading HaluEval...")
        hqa_path = f"{base_path}/halueval.json"
        if Path(hqa_path).exists():
            try:
                # Try different JSON formats
                raw = []
                with open(hqa_path) as f:
                    content = f.read().strip()
                    
                # Try parsing as JSON array first
                try:
                    data = _json.loads(content)
                    if isinstance(data, list):
                        raw = data
                    elif isinstance(data, dict):
                        # Try common keys
                        for key in ['data', 'qa', 'questions', 'examples']:
                            if key in data and isinstance(data[key], list):
                                raw = data[key]
                                break
                        if not raw:
                            # Check if dict values are lists
                            for v in data.values():
                                if isinstance(v, list) and v:
                                    raw = v
                                    break
                except:
                    # Try line-by-line JSON (this is the HaluEval format!)
                    lines = content.split('\n')
                    for line in lines:
                        line = line.strip()
                        if line:
                            try:
                                raw.append(_json.loads(line))
                            except:
                                pass
                
                print(f"  Found {len(raw)} records in JSON")
                
                _r.seed(SEED)
                _r.shuffle(raw)
                items = []
                for r in raw[:sample_size]:
                    # Extract question from dialogue_history
                    dialogue = r.get("dialogue_history", "")
                    knowledge = r.get("knowledge", "")
                    
                    # Try to extract question from dialogue
                    q = None
                    if dialogue:
                        # Look for [Human]: pattern
                        import re
                        matches = re.findall(r'\[Human\]:\s*(.+?)(?=\s*\[|\s*$)', dialogue, re.DOTALL)
                        if matches:
                            q = matches[-1].strip()  # Last human question
                    
                    # Fallback: use knowledge as question
                    if not q:
                        q = knowledge.strip()
                    
                    # Get gold (right response)
                    gold = r.get("right_response", "").strip()
                    
                    # Get hallucinated response
                    hall = r.get("hallucinated_response", "").strip()
                    
                    if q and gold:
                        items.append({
                            "question": q,
                            "gold": gold,
                            "hallucinated": hall if hall and hall != 'nan' and hall != 'None' else None,
                            "corpus": [gold]
                        })
                
                out["halueval"] = items
                print(f"  ✅ HaluEval: {len(items)} samples from JSON")
                
                # If still 0, inspect the JSON structure
                if len(items) == 0 and raw:
                    print(f"  ⚠️  First record keys: {list(raw[0].keys()) if raw else 'empty'}")
                    print("  Please check the JSON format")
                    
            except Exception as e:
                print(f"  ⚠️  HaluEval loading failed: {e}")
                import traceback
                traceback.print_exc()
                print("  Loading from HuggingFace...")
                try:
                    ds = load_dataset("pminervini/HaluEval", "qa", split="data")
                    ds = ds.shuffle(seed=SEED).select(range(min(sample_size, len(ds))))
                    out["halueval"] = [{"question": r["question"], "gold": r["right_answer"],
                                         "hallucinated": r["hallucinated_answer"],
                                         "corpus": [r["right_answer"]]} for r in ds]
                    print(f"  ✅ HaluEval HF: {len(out['halueval'])} samples")
                except Exception as e2:
                    print(f"  ❌ Failed to load HaluEval: {e2}")
        else:
            print("  HaluEval JSON not found, loading from HuggingFace...")
            try:
                ds = load_dataset("pminervini/HaluEval", "qa", split="data")
                ds = ds.shuffle(seed=SEED).select(range(min(sample_size, len(ds))))
                out["halueval"] = [{"question": r["question"], "gold": r["right_answer"],
                                     "hallucinated": r["hallucinated_answer"],
                                     "corpus": [r["right_answer"]]} for r in ds]
                print(f"  ✅ HaluEval HF: {len(out['halueval'])} samples")
            except Exception as e:
                print(f"  ❌ Failed to load HaluEval: {e}")

    print(f"\n📊 Datasets loaded: { {k: len(v) for k,v in out.items()} }")
    return out

# ── PAI COMPUTATION ───────────────────────────────────────────────────────────
def compute_pai(h_rates):
    if not h_rates or h_rates[0] == 0:
        return [1.0] * len(h_rates)
    return [round(h / h_rates[0], 4) for h in h_rates]

def compute_onset(pai_vals, threshold=PAI_THRESHOLD):
    for i, v in enumerate(pai_vals):
        if v >= threshold:
            return i + 1
    return None

# ── MAIN EXPERIMENT ───────────────────────────────────────────────────────────
def run_experiment():
    # Select config based on RUN_MODE
    if RUN_MODE == "A":
        models      = RUN_A_MODELS
        datasets    = RUN_A_DATASETS
        sample_size = RUN_A_SAMPLE_SIZE
        chain_len   = RUN_A_CHAIN_LEN
        temps       = RUN_A_TEMPS
        print("="*60)
        print(f"🚀 RUN A: New Architectures")
        print(f"Models: {list(models.keys())}")
        print(f"Datasets: {datasets} | n={sample_size} | K={chain_len}")
        print("="*60)
    elif RUN_MODE == "B":
        models      = RUN_B_MODELS
        datasets    = RUN_B_DATASETS
        sample_size = RUN_B_SAMPLE_SIZE
        chain_len   = RUN_B_CHAIN_LEN
        temps       = RUN_B_TEMPS
        print("="*60)
        print(f"🚀 RUN B: Temperature Sensitivity")
        print(f"Models: {list(models.keys())}")
        print(f"Datasets: {datasets} | n={sample_size} | K={chain_len}")
        print(f"Temperatures: {temps}")
        print("="*60)
    else:
        raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

    # Load checkpoint
    ckpt = load_master_checkpoint()
    done_keys = set(ckpt["done_keys"])
    rows = ckpt["rows"]
    records_since_save = 0

    # Load datasets
    print("\n📂 Loading datasets...")
    data = load_datasets(sample_size, datasets)
    
    # Check if any data was loaded
    total_questions = sum(len(data.get(d, [])) for d in datasets)
    if total_questions == 0:
        print("❌ No datasets loaded! Check your dataset paths.")
        print("Available paths tried:")
        for p in POSSIBLE_PATHS:
            print(f"  - {p}: {Path(p).exists()}")
        return

    # Load detector once
    detector = load_detector()

    # Total work for progress bar
    total_chains = len(models) * sum(len(data.get(d, [])) for d in datasets) * len(temps)
    pbar = tqdm(total=total_chains, desc="Total progress", unit="chain")

    for model_name, model_id in models.items():
        try:
            model, tok = load_model(model_name, model_id)
        except Exception as e:
            print(f"❌ SKIP {model_name}: {e}")
            pbar.update(sum(len(data.get(d, [])) for d in datasets) * len(temps))
            continue

        for dataset_name in datasets:
            questions = data.get(dataset_name, [])
            if not questions:
                print(f"⚠️  No questions for {dataset_name}, skipping")
                continue

            for temp in temps:
                temp_label = f"t{temp}"

                # PAI tracking for this config
                h_by_step = {k: [] for k in range(1, chain_len + 1)}

                q_pbar = tqdm(questions, desc=f"{model_name}|{dataset_name}|{temp_label}",
                              leave=False, unit="q")

                for q_idx, qdata in enumerate(q_pbar):
                    key = make_key(RUN_MODE, model_name, dataset_name, q_idx, temp)
                    if key in done_keys:
                        pbar.update(0)
                        continue

                    q         = qdata["question"]
                    gold      = qdata["gold"]
                    corpus    = qdata.get("corpus", [gold])
                    seed_resp = qdata.get("hallucinated", None)

                    try:
                        last_resp = None
                        chain_flags = []

                        for k in range(1, chain_len + 1):
                            ctx = last_resp if k > 1 else None

                            # Step 1: use seeded hallucination for HaluEval
                            if k == 1 and seed_resp and dataset_name == "halueval":
                                resp = seed_resp
                            else:
                                resp = generate_response(
                                    model, tok, model_name, q,
                                    prior_response=ctx,
                                    temperature=temp
                                )

                            h = is_hallucinated(detector, resp, gold)
                            chain_flags.append(h)
                            h_by_step[k].append(h)
                            last_resp = resp

                        # Save row
                        row_data = {
                            "run_mode":   RUN_MODE,
                            "model":      model_name,
                            "dataset":    dataset_name,
                            "q_idx":      q_idx,
                            "temperature": temp,
                            **{f"h_step_{k}": chain_flags[k-1] for k in range(1, chain_len+1)},
                            "h1":         chain_flags[0],
                            f"h{chain_len}": chain_flags[-1],
                        }
                        rows.append(row_data)

                        done_keys.add(key)
                        records_since_save += 1

                        # Save every CHECKPOINT_EVERY records
                        if records_since_save >= CHECKPOINT_EVERY:
                            ckpt["rows"] = rows
                            ckpt["done_keys"] = list(done_keys)
                            save_master_checkpoint(ckpt)
                            records_since_save = 0

                        q_pbar.set_postfix({"H": str(chain_flags)})

                    except Exception as e:
                        print(f"\n  ⚠️  Error q{q_idx}: {e}")
                        import traceback
                        traceback.print_exc()
                        continue
                    finally:
                        pbar.update(1)

                # PAI summary for this config
                h_means = [np.mean(h_by_step[k]) if h_by_step[k] else 0.0
                           for k in range(1, chain_len+1)]
                pai = compute_pai(h_means)
                onset = compute_onset(pai)

                print(f"\n── {model_name}|{dataset_name}|{temp_label} ──")
                for k in range(chain_len):
                    print(f"   Step {k+1}: H={h_means[k]:.3f}  PAI={pai[k]:.3f}")
                print(f"   PAI@{chain_len}: {pai[-1]:.3f} | Onset: {'Step '+str(onset) if onset else 'Not Reached'}")

        # Free GPU memory before next model
        del model, tok
        torch.cuda.empty_cache()
        print(f"\n✅ GPU cleared after {model_name}")
        time.sleep(1)  # Small pause before next model

    pbar.close()

    # Final checkpoint save
    ckpt["rows"] = rows
    ckpt["done_keys"] = list(done_keys)
    if RUN_MODE not in ckpt["run_modes_done"]:
        ckpt["run_modes_done"].append(RUN_MODE)
    save_master_checkpoint(ckpt)

    print(f"\n{'='*60}")
    print(f"✅ Run {RUN_MODE} complete. {len(rows)} total rows.")
    build_results_xlsx(rows)

# ── RESULTS XLSX ──────────────────────────────────────────────────────────────
def build_results_xlsx(rows):
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment

    if not rows:
        print("No rows to save.")
        return

    df = pd.DataFrame(rows)

    # Load existing workbook if exists, else create new
    if Path(MASTER_RESULTS).exists():
        wb = openpyxl.load_workbook(MASTER_RESULTS)
    else:
        wb = openpyxl.Workbook()
        if "Sheet" in wb.sheetnames:
            del wb["Sheet"]

    # Tab name
    tab_name = f"Run_{RUN_MODE}"
    if tab_name in wb.sheetnames:
        del wb[tab_name]
    ws = wb.create_sheet(tab_name)

    # Write
    def style_header(ws, df):
        for ci, col in enumerate(df.columns, 1):
            c = ws.cell(row=1, column=ci, value=col)
            c.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
            c.fill = PatternFill("solid", start_color="2E75B6")
            c.alignment = Alignment(horizontal="center")
        for ri, row in enumerate(df.itertuples(index=False), 2):
            for ci, val in enumerate(row, 1):
                ws.cell(row=ri, column=ci, value=val)
        for col in ws.columns:
            w = max(len(str(c.value or "")) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(w + 4, 40)

    style_header(ws, df)

    # PAI summary tab
    run_df = df[df["run_mode"] == RUN_MODE]
    chain_cols = sorted([c for c in df.columns if c.startswith("h_step_")])
    if chain_cols and len(run_df) > 0:
        summary = []
        group_cols = ["model", "dataset", "temperature"] if "temperature" in df.columns else ["model", "dataset"]
        
        for combo, g in run_df.groupby(group_cols):
            if not isinstance(combo, tuple):
                combo = (combo,)
            h_means = [g[c].mean() for c in chain_cols]
            pai = compute_pai(h_means)
            onset = compute_onset(pai)
            row = {
                "run_mode": RUN_MODE,
                **{group_cols[i]: combo[i] for i in range(len(combo))},
                "H1": round(h_means[0], 4) if h_means else 0,
                f"PAI@{len(chain_cols)}": round(pai[-1], 4) if pai else 0,
                "Onset_Step": onset if onset else "Not Reached",
                "n": len(g)
            }
            summary.append(row)

        summ_tab = f"PAI_Run_{RUN_MODE}"
        if summ_tab in wb.sheetnames:
            del wb[summ_tab]
        ws2 = wb.create_sheet(summ_tab)
        style_header(ws2, pd.DataFrame(summary))

    wb.save(MASTER_RESULTS)
    print(f"✅ Saved {MASTER_RESULTS} — tabs: {wb.sheetnames}")

# ── COMPLETION BEEP ───────────────────────────────────────────────────────────
def beep_success():
    try:
        from IPython.display import Audio, display
        import numpy as np
        t = np.linspace(0, 0.4, int(22050 * 0.4), False)
        wave = 0.5 * np.sin(2 * np.pi * 880 * t)
        for _ in range(3):
            display(Audio(wave, rate=22050, autoplay=True))
            time.sleep(0.6)
    except:
        print("\n🎉 ALL DONE!")

# ── RUN ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    try:
        run_experiment()
        print("\n🎉 ALL DONE!")
        beep_success()
    except Exception as e:
        print(f"\n❌ FATAL ERROR: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
# ============================================================
# SnowBallScan — Runs C, B, D, E
# Change RUN_MODE at the top to switch between runs
#
# Run C: Non-propagating baseline (Gemma-2B + Llama 3.2 3B)
#         All 3 datasets, n=200, K=5
# Run B: Temperature sensitivity (Phi-2 + TinyLlama + Qwen25)
#         TruthfulQA + HaluEval, n=50, K=5, temps=[0.0, 0.3, 0.7]
# Run D: Mitigation (Gemma-2B + Llama 3.2 3B)
#         All 3 datasets, n=200, K=5, BM25 at steps 2 and 4
# Run E: Extended chains (Gemma-2B + Llama 3.2 3B)
#         All 3 datasets, n=100, K=10
#
# SHARED:
#   Master checkpoint: snowballscan_v2_master.json
#   Master results:    snowballscan_v2_master.xlsx
#   Saves every 10 records
#   Resume bug fixed — reloads h_by_step from checkpoint
# ============================================================

import os, json, random, warnings, re, time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

# ── CONFIG — CHANGE THIS ──────────────────────────────────────────────────────
RUN_MODE = "B"   # "C", "B", "D", or "E"

# ── RUN SETTINGS ──────────────────────────────────────────────────────────────
CONFIGS = {
    "C": {
        "name":        "Non-Propagating Baseline",
        "models":      {"gemma2b": "google/gemma-2b",
                        "llama32_3b": "meta-llama/Llama-3.2-3B"},
        "datasets":    ["truthfulqa", "freshqa", "halueval"],
        "sample_size": 200,
        "chain_len":   5,
        "temps":       [0.0],
        "mode":        "non_propagating",
    },
    "B": {
        "name":        "Temperature Sensitivity",
        "models":      {"phi2":     "microsoft/phi-2",
                        "tinyllama":"TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                        "qwen25":   "Qwen/Qwen2.5-1.5B-Instruct"},
        "datasets":    ["truthfulqa", "halueval"],
        "sample_size": 50,
        "chain_len":   5,
        "temps":       [0.0, 0.3, 0.7],
        "mode":        "temperature",
    },
    "D": {
        "name":        "Mitigation",
        "models":      {"gemma2b": "google/gemma-2b",
                        "llama32_3b": "meta-llama/Llama-3.2-3B"},
        "datasets":    ["truthfulqa", "freshqa", "halueval"],
        "sample_size": 200,
        "chain_len":   5,
        "temps":       [0.0],
        "mode":        "mitigation",
    },
    "E": {
        "name":        "Extended K=10 Chains",
        "models":      {"gemma2b": "google/gemma-2b",
                        "llama32_3b": "meta-llama/Llama-3.2-3B"},
        "datasets":    ["truthfulqa", "freshqa", "halueval"],
        "sample_size": 100,
        "chain_len":   10,
        "temps":       [0.0],
        "mode":        "propagating",
    },
}

cfg            = CONFIGS[RUN_MODE]
MODELS         = cfg["models"]
DATASETS       = cfg["datasets"]
SAMPLE_SIZE    = cfg["sample_size"]
CHAIN_LEN      = cfg["chain_len"]
TEMPS          = cfg["temps"]
EXPERIMENT_MODE= cfg["mode"]

SEED              = 42
PAI_THRESHOLD     = 1.5
MASTER_CHECKPOINT = "snowballscan_v2_master.json"
MASTER_RESULTS    = "snowballscan_v2_master.xlsx"
CHECKPOINT_EVERY  = 10
BASE              = "/kaggle/input/datasets/samuelstephen77/snowballscan-dataset"

random.seed(SEED)
np.random.seed(SEED)

# ── HF TOKEN ─────────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login as hf_login

secrets   = UserSecretsClient()
hf_token  = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"]                = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
os.environ["HF_DATASETS_OFFLINE"]    = "1"

hf_login(token=hf_token, add_to_git_credential=False)
print("✅ HF login OK")

# ── INSTALL ───────────────────────────────────────────────────────────────────
print("Installing dependencies...")
os.system("pip install -q rank_bm25 sentence-transformers datasets openpyxl tqdm")
print("✅ Dependencies ready\n")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
from IPython.display import Audio, display

# ── BEEPS ─────────────────────────────────────────────────────────────────────
def beep_success():
    t = np.linspace(0, 0.4, int(22050*0.4), False)
    wave = 0.5 * np.sin(2*np.pi*880*t)
    for _ in range(3):
        display(Audio(wave, rate=22050, autoplay=True))
        time.sleep(0.6)

def beep_error():
    t = np.linspace(0, 0.5, int(22050*0.5), False)
    wave = 0.5 * np.sin(2*np.pi*220*t)
    for _ in range(5):
        display(Audio(wave, rate=22050, autoplay=True))
        time.sleep(0.8)

# ── CHECKPOINT ────────────────────────────────────────────────────────────────
def load_checkpoint():
    if Path(MASTER_CHECKPOINT).exists():
        with open(MASTER_CHECKPOINT) as f:
            data = json.load(f)
        print(f"✅ Resumed: {len(data['done_keys'])} chains, "
              f"{len(data['rows'])} rows")
        return data
    print("Fresh start")
    return {"rows": [], "done_keys": [], "run_modes_done": []}

def save_checkpoint(data):
    with open(MASTER_CHECKPOINT, "w") as f:
        json.dump(data, f, indent=2)

def make_key(run, model, dataset, q_idx, temp=0.0):
    return f"{run}|{model}|{dataset}|{q_idx}|t{temp}"

# ── RELOAD h_by_step FROM CHECKPOINT (fixes resume bug) ──────────────────────
def reload_h_by_step(rows, run_mode, model_name, dataset_name,
                     temp, chain_len):
    """Rebuild per-step hallucination lists from saved rows so
    PAI summary is correct even after a session restart."""
    h = defaultdict(list)
    for r in rows:
        if (r.get("run_mode") == run_mode and
                r.get("model") == model_name and
                r.get("dataset") == dataset_name and
                r.get("temperature", 0.0) == temp):
            for k in range(1, chain_len+1):
                col = f"h_step_{k}"
                if col in r:
                    h[k].append(r[col])
    return h

# ── PROMPT TEMPLATES ──────────────────────────────────────────────────────────
def format_prompt(model_name, question, prior_response=None):
    if prior_response:
        ctx = prior_response[:120].strip()
        hint = (f"Your previous answer was: \"{ctx}\". "
                f"Now answer again more carefully.")
    else:
        hint = None

    if model_name == "tinyllama":
        sys = ("You are a factual assistant. "
               "Answer questions with a single sentence.")
        usr = (f"{hint}\n\nQuestion: {question}\nAnswer:"
               if hint else f"Question: {question}\nAnswer:")
        return (f"<|system|>\n{sys}</s>\n"
                f"<|user|>\n{usr}</s>\n<|assistant|>\n")

    elif model_name == "phi2":
        if hint:
            return (f"Instruct: {hint} Answer in one sentence.\n"
                    f"Question: {question}\nOutput:")
        return (f"Instruct: Answer in one sentence.\n"
                f"Question: {question}\nOutput:")

    elif model_name == "qwen25":
        sys = ("You are a factual assistant. "
               "Answer each question with one clear sentence.")
        usr = f"{hint}\n\nQuestion: {question}" if hint else question
        return (f"<|im_start|>system\n{sys}<|im_end|>\n"
                f"<|im_start|>user\n{usr}<|im_end|>\n"
                f"<|im_start|>assistant\n")

    elif model_name == "gemma2b":
        if hint:
            return (f"<start_of_turn>user\n{hint}\n\n"
                    f"Question: {question}<end_of_turn>\n"
                    f"<start_of_turn>model\n")
        return (f"<start_of_turn>user\n"
                f"Question: {question}<end_of_turn>\n"
                f"<start_of_turn>model\n")

    elif model_name == "llama32_3b":
        sys = ("You are a factual assistant. "
               "Answer each question with one clear sentence.")
        usr = (f"{hint}\n\nQuestion: {question}"
               if hint else f"Question: {question}")
        return (f"<|begin_of_text|>"
                f"<|start_header_id|>system<|end_header_id|>\n"
                f"{sys}<|eot_id|>"
                f"<|start_header_id|>user<|end_header_id|>\n"
                f"{usr}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n")

    else:
        if hint:
            return f"{hint}\nQuestion: {question}\nAnswer:"
        return f"Question: {question}\nAnswer:"

# ── REPETITION DETECTOR ───────────────────────────────────────────────────────
def is_repetition_loop(text):
    if not text or len(text.strip()) < 5:
        return True
    words = text.lower().split()
    if len(words) > 10:
        from collections import Counter
        trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
        if Counter(trigrams).most_common(1)[0][1] >= 4:
            return True
    return False

# ── MODEL LOADER ──────────────────────────────────────────────────────────────
def load_model(model_name, model_id):
    print(f"\nLoading {model_name}...")
    torch.manual_seed(SEED)
    config = AutoConfig.from_pretrained(
        model_id, trust_remote_code=True, token=hf_token)
    if not getattr(config, 'pad_token_id', None):
        config.pad_token_id = 2
    model = AutoModelForCausalLM.from_pretrained(
        model_id, config=config,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        token=hf_token)
    tok = AutoTokenizer.from_pretrained(
        model_id, trust_remote_code=True, token=hf_token)
    if tok.pad_token is None:
        tok.pad_token     = tok.eos_token
        tok.pad_token_id  = tok.eos_token_id
    if not getattr(model.config, 'pad_token_id', None):
        model.config.pad_token_id = tok.pad_token_id
    model.eval()
    mem = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
    print(f"  ✅ {model_name} ready | GPU: {mem:.2f}GB")
    return model, tok

# ── GENERATION ────────────────────────────────────────────────────────────────
def generate_response(model, tok, model_name, question,
                      prior_response=None, temperature=0.0,
                      max_new_tokens=100):
    prompt  = format_prompt(model_name, question, prior_response)
    inputs  = tok(prompt, return_tensors="pt", truncation=True,
                  max_length=768, padding=False).to(model.device)
    do_sample = temperature > 0.0
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            repetition_penalty=1.3,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id)
    new   = out[0][inputs["input_ids"].shape[1]:]
    text  = tok.decode(new, skip_special_tokens=True).strip()
    for sep in [". ", ".\n", "\n"]:
        if sep in text:
            text = text.split(sep)[0].strip() + "."
            break
    if is_repetition_loop(text):
        inputs2 = tok(format_prompt(model_name, question, None),
                      return_tensors="pt", truncation=True,
                      max_length=512, padding=False).to(model.device)
        with torch.no_grad():
            out2 = model.generate(
                **inputs2, max_new_tokens=80,
                do_sample=False, temperature=1.0,
                repetition_penalty=1.8,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id)
        new2  = out2[0][inputs2["input_ids"].shape[1]:]
        text2 = tok.decode(new2, skip_special_tokens=True).strip()
        for sep in [". ", ".\n", "\n"]:
            if sep in text2:
                text2 = text2.split(sep)[0].strip() + "."
                break
        if not is_repetition_loop(text2):
            text = text2
    return text if text else "No answer generated."

# ── DETECTION ─────────────────────────────────────────────────────────────────
def load_detector():
    print("Loading DeBERTa detector...")
    m = CrossEncoder("cross-encoder/nli-deberta-v3-base")
    print("  ✅ Detector ready")
    return m

def is_hallucinated(detector, response, gold):
    if not response or not gold:
        return 1
    try:
        scores  = detector.predict([(gold, response)])
        e_score = (float(scores[0][2]) if scores[0].shape[0]==3
                   else float(scores[0][-1]))
    except:
        e_score = 0.0
    overlap = (len(set(gold.lower().split()) &
                   set(response.lower().split())) /
               max(len(set(gold.lower().split())), 1))
    return 0 if (e_score > 0.4 or overlap > 0.3) else 1

# ── BM25 MITIGATION ───────────────────────────────────────────────────────────
def build_bm25(corpus):
    toks = [d.lower().split() for d in corpus if d]
    return BM25Okapi(toks) if toks else None

def retrieve_correction(bm25, corpus, query):
    if not bm25 or not corpus:
        return corpus[0] if corpus else ""
    return corpus[int(np.argmax(bm25.get_scores(query.lower().split())))]

# ── PAI ───────────────────────────────────────────────────────────────────────
def compute_pai(h_rates):
    if not h_rates or h_rates[0] == 0:
        return [1.0]*len(h_rates)
    return [round(h/h_rates[0], 4) for h in h_rates]

def compute_onset(pai_vals, threshold=PAI_THRESHOLD):
    for i, v in enumerate(pai_vals):
        if v >= threshold:
            return i+1
    return None

# ── DATASET LOADER ────────────────────────────────────────────────────────────
def load_datasets(sample_size, dataset_names):
    import pandas as _pd, json as _json, random as _r
    out = {}

    if "truthfulqa" in dataset_names:
        print("Loading TruthfulQA...")
        df = _pd.read_csv(f"{BASE}/TruthfulQA.csv")
        df.columns = [c.strip() for c in df.columns]
        qc = next(c for c in df.columns if 'question' in c.lower())
        ac = next(c for c in df.columns
                  if 'best' in c.lower() and 'answer' in c.lower())
        cc = next((c for c in df.columns
                   if 'correct' in c.lower() and 'answer' in c.lower()),None)
        df = df[df[qc].notna()].sample(
            min(sample_size, len(df)), random_state=SEED)
        items = []
        for _, row in df.iterrows():
            q    = str(row[qc]).strip()
            gold = str(row[ac]).strip()
            corp = ([a.strip() for a in str(row[cc]).split(';')
                     if a.strip() and a.strip()!='nan']
                    if cc else [gold])
            if q and gold:
                items.append({"question":q,"gold":gold,
                              "corpus":corp or [gold]})
        out["truthfulqa"] = items
        print(f"  ✅ TruthfulQA: {len(items)}")

    if "freshqa" in dataset_names:
        print("Loading FreshQA...")
        df = _pd.read_csv(f"{BASE}/freshqa.csv", skiprows=2, header=None)
        cols = (['id','split','question','effective_year','next_review',
                 'false_premise','num_hops','fact_type','source'] +
                [f'answer_{i}' for i in range(10)] + ['note'])
        df.columns = cols[:len(df.columns)]
        df = df[df['question'].notna()]
        df = df[df['false_premise'].astype(str).str.upper()!='TRUE']
        df = df[df['question']!='question'].reset_index(drop=True)
        df = df.sample(min(sample_size, len(df)), random_state=SEED)
        items = []
        for _, row in df.iterrows():
            q = str(row['question']).strip()
            ans = [str(row[f'answer_{i}']).strip() for i in range(10)
                   if str(row.get(f'answer_{i}','')).strip()
                   not in ['','nan','NaN']]
            if q and ans:
                items.append({"question":q,"gold":ans[0],"corpus":ans})
        out["freshqa"] = items
        print(f"  ✅ FreshQA: {len(items)}")

    if "halueval" in dataset_names:
        print("Loading HaluEval...")
        raw = []
        with open(f"{BASE}/halueval.json") as f:
            for line in f:
                line = line.strip()
                if line:
                    try: raw.append(_json.loads(line))
                    except: pass
        print(f"  Raw lines: {len(raw)}")
        items = []
        for r in raw[:sample_size]:
            q    = str(r.get("dialogue_history","")).strip()
            gold = str(r.get("right_response","")).strip()
            hall = str(r.get("hallucinated_response","")).strip()
            if q and gold:
                items.append({"question":q,"gold":gold,
                              "hallucinated":hall if hall else None,
                              "corpus":[gold]})
        out["halueval"] = items
        print(f"  ✅ HaluEval: {len(items)}")

    return out

# ── MAIN EXPERIMENT ───────────────────────────────────────────────────────────
def run_experiment():
    print(f"\n{'='*60}")
    print(f"RUN {RUN_MODE}: {cfg['name']}")
    print(f"Models: {list(MODELS.keys())}")
    print(f"Datasets: {DATASETS} | n={SAMPLE_SIZE} | K={CHAIN_LEN}")
    print(f"Temps: {TEMPS} | Mode: {EXPERIMENT_MODE}")
    print(f"{'='*60}\n")

    ckpt         = load_checkpoint()
    done_keys    = set(ckpt["done_keys"])
    rows         = ckpt["rows"]
    n_since_save = 0

    data     = load_datasets(SAMPLE_SIZE, DATASETS)
    detector = load_detector()

    total = (len(MODELS) *
             sum(len(data.get(d,[])) for d in DATASETS) *
             len(TEMPS))
    pbar = tqdm(total=total, desc="Overall", unit="q")

    for model_name, model_id in MODELS.items():
        try:
            model, tok = load_model(model_name, model_id)
        except Exception as e:
            print(f"❌ SKIP {model_name}: {e}")
            pbar.update(
                sum(len(data.get(d,[])) for d in DATASETS)*len(TEMPS))
            continue

        for dataset_name in DATASETS:
            questions = data.get(dataset_name, [])
            if not questions:
                continue

            for temp in TEMPS:
                # ── Fix: reload h_by_step from checkpoint ────────────────
                h_by_step = reload_h_by_step(
                    rows, RUN_MODE, model_name,
                    dataset_name, temp, CHAIN_LEN)

                q_pbar = tqdm(
                    questions,
                    desc=f"{model_name}|{dataset_name}|t{temp}",
                    leave=False, unit="q")

                for q_idx, qdata in enumerate(q_pbar):
                    key = make_key(RUN_MODE, model_name,
                                   dataset_name, q_idx, temp)
                    if key in done_keys:
                        continue

                    q         = qdata["question"]
                    gold      = qdata["gold"]
                    corpus    = qdata.get("corpus", [gold])
                    seed_resp = qdata.get("hallucinated", None)

                    try:
                        bm25        = build_bm25(corpus)
                        chain_flags = []
                        last_resp   = None

                        for k in range(1, CHAIN_LEN+1):
                            # Context passing depends on mode
                            if EXPERIMENT_MODE == "non_propagating":
                                ctx = None  # no context ever
                            else:
                                ctx = last_resp if k > 1 else None

                            # Step 1 seeding for HaluEval
                            if (k == 1 and seed_resp
                                    and dataset_name == "halueval"
                                    and EXPERIMENT_MODE != "non_propagating"):
                                resp = seed_resp
                            else:
                                resp = generate_response(
                                    model, tok, model_name, q,
                                    prior_response=ctx,
                                    temperature=temp)

                            # Mitigation at steps 2 and 4
                            if (EXPERIMENT_MODE == "mitigation"
                                    and k in [2, 4]):
                                h_check = is_hallucinated(
                                    detector, resp, gold)
                                if h_check == 1:
                                    correction = retrieve_correction(
                                        bm25, corpus, q)
                                    if correction:
                                        resp = correction

                            h = is_hallucinated(detector, resp, gold)
                            chain_flags.append(h)
                            h_by_step[k].append(h)
                            last_resp = resp

                        rows.append({
                            "run_mode":    RUN_MODE,
                            "model":       model_name,
                            "dataset":     dataset_name,
                            "q_idx":       q_idx,
                            "temperature": temp,
                            "exp_mode":    EXPERIMENT_MODE,
                            **{f"h_step_{k}": chain_flags[k-1]
                               for k in range(1, CHAIN_LEN+1)},
                            "h1": chain_flags[0],
                            "h_final": chain_flags[-1],
                        })

                        done_keys.add(key)
                        n_since_save += 1

                        if n_since_save >= CHECKPOINT_EVERY:
                            ckpt["rows"]      = rows
                            ckpt["done_keys"] = list(done_keys)
                            save_checkpoint(ckpt)
                            n_since_save = 0

                        q_pbar.set_postfix(
                            {"H": str(chain_flags),
                             "saved": len(rows)})
                        pbar.update(1)

                    except Exception as e:
                        print(f"\n  ⚠️  q{q_idx} error: {e}")
                        pbar.update(1)
                        continue

                # PAI summary (correct — loaded from checkpoint)
                h_means = [np.mean(h_by_step[k]) if h_by_step[k] else 0.0
                           for k in range(1, CHAIN_LEN+1)]
                pai   = compute_pai(h_means)
                onset = compute_onset(pai)

                print(f"\n── {model_name}|{dataset_name}|t{temp} "
                      f"[{EXPERIMENT_MODE}] ──")
                for k in range(CHAIN_LEN):
                    print(f"   Step {k+1}: H={h_means[k]:.3f}  "
                          f"PAI={pai[k]:.3f}")
                print(f"   PAI@{CHAIN_LEN}: {pai[-1]:.3f} | "
                      f"Onset: {'Step '+str(onset) if onset else 'NR'}")

        del model, tok
        torch.cuda.empty_cache()
        print(f"\n✅ GPU cleared: {model_name}")

    pbar.close()

    ckpt["rows"]      = rows
    ckpt["done_keys"] = list(done_keys)
    if RUN_MODE not in ckpt.get("run_modes_done",[]):
        ckpt.setdefault("run_modes_done",[]).append(RUN_MODE)
    save_checkpoint(ckpt)
    print(f"\n{'='*60}")
    print(f"Run {RUN_MODE} complete | {len(rows)} total rows")
    build_results_xlsx(rows)

# ── RESULTS XLSX ──────────────────────────────────────────────────────────────
def build_results_xlsx(rows):
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment

    if not rows:
        print("No rows.")
        return

    df = pd.DataFrame(rows)

    wb = (openpyxl.load_workbook(MASTER_RESULTS)
          if Path(MASTER_RESULTS).exists()
          else openpyxl.Workbook())
    if "Sheet" in wb.sheetnames:
        del wb["Sheet"]

    def write_tab(name, data_df):
        if name in wb.sheetnames:
            del wb[name]
        ws = wb.create_sheet(name)
        for ci, col in enumerate(data_df.columns, 1):
            c = ws.cell(row=1, column=ci, value=col)
            c.font = Font(bold=True, color="FFFFFF",
                          name="Arial", size=10)
            c.fill = PatternFill("solid", start_color="2E75B6")
            c.alignment = Alignment(horizontal="center")
        for ri, row in enumerate(data_df.itertuples(index=False), 2):
            for ci, val in enumerate(row, 1):
                ws.cell(row=ri, column=ci, value=val)
        for col in ws.columns:
            w = max(len(str(c.value or "")) for c in col)
            ws.column_dimensions[
                col[0].column_letter].width = min(w+4, 40)

    # Raw tab for this run
    run_df = df[df["run_mode"]==RUN_MODE]
    write_tab(f"Raw_{RUN_MODE}", run_df)

    # PAI summary tab
    chain_cols = sorted([c for c in df.columns
                         if c.startswith("h_step_")])
    summary = []
    for (m, ds, temp, em), g in run_df.groupby(
            ["model","dataset","temperature","exp_mode"]):
        h_means = [g[c].mean() for c in chain_cols if c in g.columns]
        pai     = compute_pai(h_means)
        onset   = compute_onset(pai)
        summary.append({
            "run_mode": RUN_MODE, "model": m,
            "dataset": ds, "temperature": temp,
            "exp_mode": em,
            "H1": round(h_means[0], 4),
            f"PAI@{CHAIN_LEN}": round(pai[-1], 4),
            **{f"PAI@{i+1}": round(pai[i], 4)
               for i in range(len(pai))},
            "Onset": onset if onset else "NR",
            "n": len(g)
        })
    write_tab(f"PAI_{RUN_MODE}", pd.DataFrame(summary))

    wb.save(MASTER_RESULTS)
    print(f"✅ Saved {MASTER_RESULTS} | Tabs: {wb.sheetnames}")

# ── ENTRY POINT ───────────────────────────────────────────────────────────────
try:
    run_experiment()
    print("\n🎉 ALL DONE!")
    beep_success()
except Exception as e:
    print(f"\n❌ FATAL ERROR: {e}")
    import traceback; traceback.print_exc()
    beep_error()

In [ ]:
import subprocess
result = subprocess.run(
    ['python3', '-c', '''
import os
os.environ["HF_DATASETS_OFFLINE"] = "0"
os.environ["HF_HUB_OFFLINE"] = "0"
from datasets import load_dataset
import json
ds = load_dataset("pminervini/HaluEval", "qa", split="data")
with open("/kaggle/working/halueval_qa.json", "w") as f:
    for r in ds:
        f.write(json.dumps(r) + "\\n")
print(f"Done: {len(ds)} records")
print(ds[0])
'''],
    env={**os.environ, 'HF_DATASETS_OFFLINE': '0', 'HF_HUB_OFFLINE': '0'},
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [1]:
# ============================================================
# SnowBallScan v3 — Corrected HaluEval QA Split
# New checkpoint: snowballscan_v3_master.json
# New results:    snowballscan_v3_master.xlsx
#
# Run A2: Propagating chains — Gemma-2B + Llama 3.2 3B
#          HaluEval QA only, n=200, K=5
# Run C2: Non-propagating — Gemma-2B + Llama 3.2 3B
#          HaluEval QA only, n=200, K=5
# Run B2: Temperature sensitivity — Phi-2 + TinyLlama + Qwen25
#          HaluEval QA only, n=50, K=5, temps=[0.0, 0.3, 0.7]
# Run D:  Mitigation — Gemma-2B + Llama 3.2 3B
#          All 3 datasets, n=200, K=5
# Run E:  K=10 extended — Gemma-2B + Llama 3.2 3B
#          All 3 datasets, n=100, K=10
#
# KEY CHECK: HaluEval H(1) should be ~0.415 (same as v10)
# If you see ~0.065, wrong split — stop immediately
# ============================================================

import os, json, random, warnings, re, time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore")

# ── CHANGE THIS ───────────────────────────────────────────────────────────────
RUN_MODE = "E"   # A2 → C2 → B2 → D → E

# ── RUN CONFIGS ───────────────────────────────────────────────────────────────
CONFIGS = {
    "A2": {
        "name":        "Propagating — HaluEval QA Fix",
        "models":      {"gemma2b":    "google/gemma-2b",
                        "llama32_3b": "meta-llama/Llama-3.2-3B"},
        "datasets":    ["halueval"],
        "sample_size": 200,
        "chain_len":   5,
        "temps":       [0.0],
        "mode":        "propagating",
    },
    "C2": {
        "name":        "Non-Propagating — HaluEval QA Fix",
        "models":      {"gemma2b":    "google/gemma-2b",
                        "llama32_3b": "meta-llama/Llama-3.2-3B"},
        "datasets":    ["halueval"],
        "sample_size": 200,
        "chain_len":   5,
        "temps":       [0.0],
        "mode":        "non_propagating",
    },
    "B2": {
        "name":        "Temperature Sensitivity — HaluEval QA Fix",
        "models":      {"phi2":      "microsoft/phi-2",
                        "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
                        "qwen25":    "Qwen/Qwen2.5-1.5B-Instruct"},
        "datasets":    ["halueval"],
        "sample_size": 50,
        "chain_len":   5,
        "temps":       [0.0, 0.3, 0.7],
        "mode":        "temperature",
    },
    "D": {
        "name":        "Mitigation",
        "models":      {"gemma2b":    "google/gemma-2b",
                        "llama32_3b": "meta-llama/Llama-3.2-3B"},
        "datasets":    ["truthfulqa", "freshqa", "halueval"],
        "sample_size": 200,
        "chain_len":   5,
        "temps":       [0.0],
        "mode":        "mitigation",
    },
    "E": {
        "name":        "Extended K=10 Chains",
        "models":      {"gemma2b":    "google/gemma-2b",
                        "llama32_3b": "meta-llama/Llama-3.2-3B"},
        "datasets":    ["truthfulqa", "freshqa", "halueval"],
        "sample_size": 100,
        "chain_len":   10,
        "temps":       [0.0],
        "mode":        "propagating",
    },
}

cfg             = CONFIGS[RUN_MODE]
MODELS          = cfg["models"]
DATASETS        = cfg["datasets"]
SAMPLE_SIZE     = cfg["sample_size"]
CHAIN_LEN       = cfg["chain_len"]
TEMPS           = cfg["temps"]
EXPERIMENT_MODE = cfg["mode"]

SEED               = 42
PAI_THRESHOLD      = 1.5
MASTER_CHECKPOINT  = "snowballscan_v3_master.json"
MASTER_RESULTS     = "snowballscan_v3_master.xlsx"
CHECKPOINT_EVERY   = 20
BASE               = "/kaggle/input/datasets/samuelstephen77/snowballscan-dataset"
HALUEVAL_QA_FILE   = f"{BASE}/halueval_qa.json"  # correct QA split

random.seed(SEED)
np.random.seed(SEED)

# ── HF TOKEN ─────────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login as hf_login

secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"]                = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
os.environ["HF_DATASETS_OFFLINE"]    = "1"

hf_login(token=hf_token, add_to_git_credential=False)
print("✅ HF login OK")

# ── INSTALL ───────────────────────────────────────────────────────────────────
print("Installing dependencies...")
os.system("pip install -q rank_bm25 sentence-transformers datasets openpyxl tqdm")
print("✅ Dependencies ready\n")

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm
from IPython.display import Audio, display

# ── BEEPS ─────────────────────────────────────────────────────────────────────
def beep_success():
    t = np.linspace(0, 0.4, int(22050*0.4), False)
    wave = 0.5 * np.sin(2*np.pi*880*t)
    for _ in range(3):
        display(Audio(wave, rate=22050, autoplay=True))
        time.sleep(0.6)

def beep_error():
    t = np.linspace(0, 0.5, int(22050*0.5), False)
    wave = 0.5 * np.sin(2*np.pi*220*t)
    for _ in range(5):
        display(Audio(wave, rate=22050, autoplay=True))
        time.sleep(0.8)

# ── CHECKPOINT ────────────────────────────────────────────────────────────────
def load_checkpoint():
    if Path(MASTER_CHECKPOINT).exists():
        with open(MASTER_CHECKPOINT) as f:
            data = json.load(f)
        print(f"✅ Resumed: {len(data['done_keys'])} chains, "
              f"{len(data['rows'])} rows")
        return data
    print("Fresh start — no checkpoint found")
    return {"rows": [], "done_keys": [], "run_modes_done": []}

def save_checkpoint(data):
    with open(MASTER_CHECKPOINT, "w") as f:
        json.dump(data, f, indent=2)

def make_key(run, model, dataset, q_idx, temp=0.0):
    return f"{run}|{model}|{dataset}|{q_idx}|t{temp}"

# ── RELOAD h_by_step FROM CHECKPOINT (resume bug fix) ────────────────────────
def reload_h_by_step(rows, run_mode, model_name,
                     dataset_name, temp, chain_len):
    h = defaultdict(list)
    for r in rows:
        if (r.get("run_mode") == run_mode and
                r.get("model") == model_name and
                r.get("dataset") == dataset_name and
                r.get("temperature", 0.0) == temp):
            for k in range(1, chain_len+1):
                col = f"h_step_{k}"
                if col in r:
                    h[k].append(r[col])
    return h

# ── PROMPT TEMPLATES ──────────────────────────────────────────────────────────
def format_prompt(model_name, question, prior_response=None):
    if prior_response:
        ctx  = prior_response[:120].strip()
        hint = (f"Your previous answer was: \"{ctx}\". "
                f"Now answer again more carefully.")
    else:
        hint = None

    if model_name == "tinyllama":
        sys = ("You are a factual assistant. "
               "Answer questions with a single sentence.")
        usr = (f"{hint}\n\nQuestion: {question}\nAnswer:"
               if hint else f"Question: {question}\nAnswer:")
        return (f"<|system|>\n{sys}</s>\n"
                f"<|user|>\n{usr}</s>\n<|assistant|>\n")

    elif model_name == "phi2":
        if hint:
            return (f"Instruct: {hint} Answer in one sentence.\n"
                    f"Question: {question}\nOutput:")
        return (f"Instruct: Answer in one sentence.\n"
                f"Question: {question}\nOutput:")

    elif model_name == "qwen25":
        sys = ("You are a factual assistant. "
               "Answer each question with one clear sentence.")
        usr = f"{hint}\n\nQuestion: {question}" if hint else question
        return (f"<|im_start|>system\n{sys}<|im_end|>\n"
                f"<|im_start|>user\n{usr}<|im_end|>\n"
                f"<|im_start|>assistant\n")

    elif model_name == "gemma2b":
        if hint:
            return (f"<start_of_turn>user\n{hint}\n\n"
                    f"Question: {question}<end_of_turn>\n"
                    f"<start_of_turn>model\n")
        return (f"<start_of_turn>user\n"
                f"Question: {question}<end_of_turn>\n"
                f"<start_of_turn>model\n")

    elif model_name == "llama32_3b":
        sys = ("You are a factual assistant. "
               "Answer each question with one clear sentence.")
        usr = (f"{hint}\n\nQuestion: {question}"
               if hint else f"Question: {question}")
        return (f"<|begin_of_text|>"
                f"<|start_header_id|>system<|end_header_id|>\n"
                f"{sys}<|eot_id|>"
                f"<|start_header_id|>user<|end_header_id|>\n"
                f"{usr}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n")

    else:
        if hint:
            return f"{hint}\nQuestion: {question}\nAnswer:"
        return f"Question: {question}\nAnswer:"

# ── REPETITION DETECTOR ───────────────────────────────────────────────────────
def is_repetition_loop(text):
    if not text or len(text.strip()) < 5:
        return True
    words = text.lower().split()
    if len(words) > 10:
        from collections import Counter
        trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
        if Counter(trigrams).most_common(1)[0][1] >= 4:
            return True
    return False

# ── MODEL LOADER ──────────────────────────────────────────────────────────────
def load_model(model_name, model_id):
    print(f"\nLoading {model_name}...")
    torch.manual_seed(SEED)
    config = AutoConfig.from_pretrained(
        model_id, trust_remote_code=True, token=hf_token)
    if not getattr(config, 'pad_token_id', None):
        config.pad_token_id = 2
    model = AutoModelForCausalLM.from_pretrained(
        model_id, config=config,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        token=hf_token)
    tok = AutoTokenizer.from_pretrained(
        model_id, trust_remote_code=True, token=hf_token)
    if tok.pad_token is None:
        tok.pad_token    = tok.eos_token
        tok.pad_token_id = tok.eos_token_id
    if not getattr(model.config, 'pad_token_id', None):
        model.config.pad_token_id = tok.pad_token_id
    model.eval()
    mem = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
    print(f"  ✅ {model_name} ready | GPU: {mem:.2f}GB")
    return model, tok

# ── GENERATION ────────────────────────────────────────────────────────────────
def generate_response(model, tok, model_name, question,
                      prior_response=None, temperature=0.0,
                      max_new_tokens=100):
    prompt  = format_prompt(model_name, question, prior_response)
    inputs  = tok(prompt, return_tensors="pt", truncation=True,
                  max_length=768, padding=False).to(model.device)
    do_sample = temperature > 0.0
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            repetition_penalty=1.3,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id)
    new  = out[0][inputs["input_ids"].shape[1]:]
    text = tok.decode(new, skip_special_tokens=True).strip()
    for sep in [". ", ".\n", "\n"]:
        if sep in text:
            text = text.split(sep)[0].strip() + "."
            break
    if is_repetition_loop(text):
        inputs2 = tok(format_prompt(model_name, question, None),
                      return_tensors="pt", truncation=True,
                      max_length=512, padding=False).to(model.device)
        with torch.no_grad():
            out2 = model.generate(
                **inputs2, max_new_tokens=80,
                do_sample=False, temperature=1.0,
                repetition_penalty=1.8,
                pad_token_id=tok.pad_token_id,
                eos_token_id=tok.eos_token_id)
        new2  = out2[0][inputs2["input_ids"].shape[1]:]
        text2 = tok.decode(new2, skip_special_tokens=True).strip()
        for sep in [". ", ".\n", "\n"]:
            if sep in text2:
                text2 = text2.split(sep)[0].strip() + "."
                break
        if not is_repetition_loop(text2):
            text = text2
    return text if text else "No answer generated."

# ── DETECTION ─────────────────────────────────────────────────────────────────
def load_detector():
    print("Loading DeBERTa detector...")
    m = CrossEncoder("cross-encoder/nli-deberta-v3-base")
    print("  ✅ Detector ready")
    return m

def is_hallucinated(detector, response, gold):
    if not response or not gold:
        return 1
    try:
        scores  = detector.predict([(gold, response)])
        e_score = (float(scores[0][2]) if scores[0].shape[0]==3
                   else float(scores[0][-1]))
    except:
        e_score = 0.0
    overlap = (len(set(gold.lower().split()) &
                   set(response.lower().split())) /
               max(len(set(gold.lower().split())), 1))
    return 0 if (e_score > 0.4 or overlap > 0.3) else 1

# ── BM25 ──────────────────────────────────────────────────────────────────────
def build_bm25(corpus):
    toks = [d.lower().split() for d in corpus if d]
    return BM25Okapi(toks) if toks else None

def retrieve_correction(bm25, corpus, query):
    if not bm25 or not corpus:
        return corpus[0] if corpus else ""
    return corpus[int(np.argmax(bm25.get_scores(query.lower().split())))]

# ── PAI ───────────────────────────────────────────────────────────────────────
def compute_pai(h_rates):
    if not h_rates or h_rates[0] == 0:
        return [1.0]*len(h_rates)
    return [round(h/h_rates[0], 4) for h in h_rates]

def compute_onset(pai_vals, threshold=PAI_THRESHOLD):
    for i, v in enumerate(pai_vals):
        if v >= threshold:
            return i+1
    return None

# ── DATASET LOADER ────────────────────────────────────────────────────────────
def load_datasets(sample_size, dataset_names):
    import pandas as _pd, json as _json, random as _r
    out = {}

    if "truthfulqa" in dataset_names:
        print("Loading TruthfulQA...")
        df  = _pd.read_csv(f"{BASE}/TruthfulQA.csv")
        df.columns = [c.strip() for c in df.columns]
        qc  = next(c for c in df.columns if 'question' in c.lower())
        ac  = next(c for c in df.columns
                   if 'best' in c.lower() and 'answer' in c.lower())
        cc  = next((c for c in df.columns
                    if 'correct' in c.lower()
                    and 'answer' in c.lower()), None)
        df  = df[df[qc].notna()].sample(
            min(sample_size, len(df)), random_state=SEED)
        items = []
        for _, row in df.iterrows():
            q    = str(row[qc]).strip()
            gold = str(row[ac]).strip()
            corp = ([a.strip() for a in str(row[cc]).split(';')
                     if a.strip() and a.strip()!='nan']
                    if cc else [gold])
            if q and gold:
                items.append({"question":q,"gold":gold,
                              "corpus":corp or [gold]})
        out["truthfulqa"] = items
        print(f"  ✅ TruthfulQA: {len(items)}")

    if "freshqa" in dataset_names:
        print("Loading FreshQA...")
        df = _pd.read_csv(f"{BASE}/freshqa.csv", skiprows=2, header=None)
        cols = (['id','split','question','effective_year','next_review',
                 'false_premise','num_hops','fact_type','source'] +
                [f'answer_{i}' for i in range(10)] + ['note'])
        df.columns = cols[:len(df.columns)]
        df = df[df['question'].notna()]
        df = df[df['false_premise'].astype(str).str.upper()!='TRUE']
        df = df[df['question']!='question'].reset_index(drop=True)
        df = df.sample(min(sample_size, len(df)), random_state=SEED)
        items = []
        for _, row in df.iterrows():
            q   = str(row['question']).strip()
            ans = [str(row[f'answer_{i}']).strip() for i in range(10)
                   if str(row.get(f'answer_{i}','')).strip()
                   not in ['','nan','NaN']]
            if q and ans:
                items.append({"question":q,"gold":ans[0],"corpus":ans})
        out["freshqa"] = items
        print(f"  ✅ FreshQA: {len(items)}")

    if "halueval" in dataset_names:
        print("Loading HaluEval QA...")
        # ── CORRECT QA SPLIT ─────────────────────────────────────────────
        if not Path(HALUEVAL_QA_FILE).exists():
            raise FileNotFoundError(
                f"halueval_qa.json not found at {HALUEVAL_QA_FILE}\n"
                f"Upload the QA split to your Kaggle dataset first!")
        raw = []
        with open(HALUEVAL_QA_FILE) as f:
            for line in f:
                line = line.strip()
                if line:
                    try: raw.append(_json.loads(line))
                    except: pass
        print(f"  Raw lines: {len(raw)}")
        # Verify correct split
        if raw and 'question' not in raw[0]:
            raise ValueError(
                f"Wrong HaluEval split! Keys: {list(raw[0].keys())}\n"
                f"Expected 'question', 'right_answer', 'hallucinated_answer'")
        # Sample without shuffle to be safe
        items = []
        for r in raw[:sample_size]:
            q    = str(r.get("question","")).strip()
            gold = str(r.get("right_answer","")).strip()
            hall = str(r.get("hallucinated_answer","")).strip()
            if q and gold:
                items.append({"question":q,"gold":gold,
                              "hallucinated":hall if hall else None,
                              "corpus":[gold]})
        out["halueval"] = items
        print(f"  ✅ HaluEval QA: {len(items)}")
        # KEY CHECK
        print(f"  ⚠️  KEY CHECK — H(1) should be ~0.415 after first model runs")
        print(f"  ⚠️  If H(1) < 0.2 → STOP — wrong dataset split!")

    return out

# ── MAIN EXPERIMENT ───────────────────────────────────────────────────────────
def run_experiment():
    print(f"\n{'='*60}")
    print(f"RUN {RUN_MODE}: {cfg['name']}")
    print(f"Models: {list(MODELS.keys())}")
    print(f"Datasets: {DATASETS} | n={SAMPLE_SIZE} | K={CHAIN_LEN}")
    print(f"Temps: {TEMPS} | Mode: {EXPERIMENT_MODE}")
    print(f"Checkpoint every: {CHECKPOINT_EVERY} records")
    print(f"{'='*60}\n")

    ckpt         = load_checkpoint()
    done_keys    = set(ckpt["done_keys"])
    rows         = ckpt["rows"]
    n_since_save = 0

    data     = load_datasets(SAMPLE_SIZE, DATASETS)
    detector = load_detector()

    total = (len(MODELS) *
             sum(len(data.get(d,[])) for d in DATASETS) *
             len(TEMPS))
    pbar = tqdm(total=total, desc="Overall", unit="q")

    for model_name, model_id in MODELS.items():
        try:
            model, tok = load_model(model_name, model_id)
        except Exception as e:
            print(f"❌ SKIP {model_name}: {e}")
            pbar.update(
                sum(len(data.get(d,[])) for d in DATASETS)*len(TEMPS))
            continue

        for dataset_name in DATASETS:
            questions = data.get(dataset_name, [])
            if not questions:
                continue

            for temp in TEMPS:
                # Resume fix — reload existing data
                h_by_step = reload_h_by_step(
                    rows, RUN_MODE, model_name,
                    dataset_name, temp, CHAIN_LEN)

                q_pbar = tqdm(
                    questions,
                    desc=f"{model_name}|{dataset_name}|t{temp}",
                    leave=False, unit="q")

                for q_idx, qdata in enumerate(q_pbar):
                    key = make_key(RUN_MODE, model_name,
                                   dataset_name, q_idx, temp)
                    if key in done_keys:
                        continue

                    q         = qdata["question"]
                    gold      = qdata["gold"]
                    corpus    = qdata.get("corpus", [gold])
                    seed_resp = qdata.get("hallucinated", None)

                    try:
                        bm25        = build_bm25(corpus)
                        chain_flags = []
                        last_resp   = None

                        for k in range(1, CHAIN_LEN+1):
                            if EXPERIMENT_MODE == "non_propagating":
                                ctx = None
                            else:
                                ctx = last_resp if k > 1 else None

                            if (k == 1 and seed_resp
                                    and dataset_name == "halueval"
                                    and EXPERIMENT_MODE != "non_propagating"):
                                resp = seed_resp
                            else:
                                resp = generate_response(
                                    model, tok, model_name, q,
                                    prior_response=ctx,
                                    temperature=temp)

                            if (EXPERIMENT_MODE == "mitigation"
                                    and k in [2, 4]):
                                if is_hallucinated(detector, resp, gold)==1:
                                    corr = retrieve_correction(bm25,corpus,q)
                                    if corr:
                                        resp = corr

                            h = is_hallucinated(detector, resp, gold)
                            chain_flags.append(h)
                            h_by_step[k].append(h)
                            last_resp = resp

                        rows.append({
                            "run_mode":    RUN_MODE,
                            "model":       model_name,
                            "dataset":     dataset_name,
                            "q_idx":       q_idx,
                            "temperature": temp,
                            "exp_mode":    EXPERIMENT_MODE,
                            **{f"h_step_{k}": chain_flags[k-1]
                               for k in range(1, CHAIN_LEN+1)},
                            "h1":    chain_flags[0],
                            "h_final": chain_flags[-1],
                        })

                        done_keys.add(key)
                        n_since_save += 1

                        if n_since_save >= CHECKPOINT_EVERY:
                            ckpt["rows"]      = rows
                            ckpt["done_keys"] = list(done_keys)
                            save_checkpoint(ckpt)
                            n_since_save = 0

                        q_pbar.set_postfix(
                            {"H": str(chain_flags),
                             "saved": len(rows)})
                        pbar.update(1)

                    except Exception as e:
                        print(f"\n  ⚠️  q{q_idx} error: {e}")
                        pbar.update(1)
                        continue

                # PAI summary
                h_means = [np.mean(h_by_step[k]) if h_by_step[k] else 0.0
                           for k in range(1, CHAIN_LEN+1)]
                pai   = compute_pai(h_means)
                onset = compute_onset(pai)

                print(f"\n── {model_name}|{dataset_name}|t{temp} "
                      f"[{EXPERIMENT_MODE}] ──")
                for k in range(CHAIN_LEN):
                    print(f"   Step {k+1}: H={h_means[k]:.3f}  "
                          f"PAI={pai[k]:.3f}")
                print(f"   PAI@{CHAIN_LEN}: {pai[-1]:.3f} | "
                      f"Onset: {'Step '+str(onset) if onset else 'NR'}")

                # KEY CHECK ALERT
                if dataset_name == "halueval" and h_means[0] < 0.2:
                    print(f"\n   ❌ KEY CHECK FAILED!")
                    print(f"   H(1)={h_means[0]:.3f} — expected ~0.415")
                    print(f"   WRONG DATASET SPLIT — STOP THE RUN!")
                    beep_error()
                elif dataset_name == "halueval":
                    print(f"   ✅ KEY CHECK PASSED — H(1)={h_means[0]:.3f}")

        del model, tok
        torch.cuda.empty_cache()
        print(f"\n✅ GPU cleared: {model_name}")

    pbar.close()

    ckpt["rows"]      = rows
    ckpt["done_keys"] = list(done_keys)
    ckpt.setdefault("run_modes_done",[])
    if RUN_MODE not in ckpt["run_modes_done"]:
        ckpt["run_modes_done"].append(RUN_MODE)
    save_checkpoint(ckpt)

    print(f"\n{'='*60}")
    print(f"✅ Run {RUN_MODE} complete | {len(rows)} total rows")
    build_results_xlsx(rows)

# ── RESULTS XLSX ──────────────────────────────────────────────────────────────
def build_results_xlsx(rows):
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment

    if not rows:
        print("No rows.")
        return

    df = pd.DataFrame(rows)
    wb = (openpyxl.load_workbook(MASTER_RESULTS)
          if Path(MASTER_RESULTS).exists()
          else openpyxl.Workbook())
    if "Sheet" in wb.sheetnames:
        del wb["Sheet"]

    def write_tab(name, data_df):
        if name in wb.sheetnames:
            del wb[name]
        ws = wb.create_sheet(name)
        for ci, col in enumerate(data_df.columns, 1):
            c = ws.cell(row=1, column=ci, value=col)
            c.font = Font(bold=True,color="FFFFFF",name="Arial",size=10)
            c.fill = PatternFill("solid",start_color="2E75B6")
            c.alignment = Alignment(horizontal="center")
        for ri, row in enumerate(data_df.itertuples(index=False), 2):
            for ci, val in enumerate(row, 1):
                ws.cell(row=ri, column=ci, value=val)
        for col in ws.columns:
            w = max(len(str(c.value or "")) for c in col)
            ws.column_dimensions[
                col[0].column_letter].width = min(w+4,40)

    run_df = df[df["run_mode"]==RUN_MODE]
    write_tab(f"Raw_{RUN_MODE}", run_df)

    chain_cols = sorted([c for c in df.columns
                         if c.startswith("h_step_")])
    summary = []
    group_cols = [c for c in ["model","dataset","temperature","exp_mode"]
                  if c in run_df.columns]
    for keys, g in run_df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        key_dict = dict(zip(group_cols, keys))
        h_means  = [g[c].mean() for c in chain_cols if c in g.columns]
        pai      = compute_pai(h_means)
        onset    = compute_onset(pai)
        summary.append({
            "run_mode": RUN_MODE,
            **key_dict,
            "H1": round(h_means[0],4),
            f"PAI@{CHAIN_LEN}": round(pai[-1],4),
            **{f"PAI@{i+1}": round(pai[i],4) for i in range(len(pai))},
            "Onset": onset if onset else "NR",
            "n": len(g)
        })
    write_tab(f"PAI_{RUN_MODE}", pd.DataFrame(summary))

    wb.save(MASTER_RESULTS)
    print(f"✅ Saved {MASTER_RESULTS} | Tabs: {wb.sheetnames}")

# ── ENTRY POINT ───────────────────────────────────────────────────────────────
try:
    run_experiment()
    print("\n🎉 ALL DONE!")
    beep_success()
except Exception as e:
    print(f"\n❌ FATAL ERROR: {e}")
    import traceback; traceback.print_exc()
    beep_error()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HF login OK
Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

✅ Dependencies ready


RUN E: Extended K=10 Chains
Models: ['gemma2b', 'llama32_3b']
Datasets: ['truthfulqa', 'freshqa', 'halueval'] | n=100 | K=10
Temps: [0.0] | Mode: propagating
Checkpoint every: 20 records

✅ Resumed: 2790 chains, 2790 rows
Loading TruthfulQA...
  ✅ TruthfulQA: 100
Loading FreshQA...
  ✅ FreshQA: 100
Loading HaluEval QA...
  Raw lines: 10000
  ✅ HaluEval QA: 100
  ⚠️  KEY CHECK — H(1) should be ~0.415 after first model runs
  ⚠️  If H(1) < 0.2 → STOP — wrong dataset split!
Loading DeBERTa detector...


config.json: 0.00B [00:00, ?B/s]

...

llama32_3b|truthfulqa|t0.0:   0%|          | 0/100 [00:00<?, ?q/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



── llama32_3b|truthfulqa|t0.0 [propagating] ──
   Step 1: H=0.270  PAI=1.000
   Step 2: H=0.060  PAI=0.222
   Step 3: H=0.210  PAI=0.778
   Step 4: H=0.110  PAI=0.407
   Step 5: H=0.240  PAI=0.889
   Step 6: H=0.200  PAI=0.741
   Step 7: H=0.240  PAI=0.889
   Step 8: H=0.140  PAI=0.518
   Step 9: H=0.270  PAI=1.000
   Step 10: H=0.140  PAI=0.518
   PAI@10: 0.518 | Onset: NR


llama32_3b|freshqa|t0.0:   0%|          | 0/100 [00:00<?, ?q/s]

KeyboardInterrupt: 

In [2]:
import json
with open('/kaggle/working/snowballscan_v3_master.json') as f:
    ckpt = json.load(f)
print(f'Chains done: {len(ckpt["done_keys"])}')
print(f'Rows: {len(ckpt["rows"])}')

Chains done: 2870
Rows: 2870
